In [1]:
import random
from collections import Counter, defaultdict, deque

# 1) Arrays and Hashing

**The gist.** Don't search the array again — remember what you've seen in a dict, so the search becomes one lookup.

> **Think of it like…** a party where you want someone with your birthday. Asking everyone is slow; writing every birthday on a whiteboard as people arrive means each newcomer just glances at the board.

The only real question: **what key do I store so the answer is a lookup?**

| Store this | Example |
|---|---|
| the value | Two Sum, Contains Duplicate |
| a canonical form | sorted letters / letter counts (anagrams) |
| a count | Ransom Note, Top K Frequent |
| the last index | Contains Duplicate II |
| the index itself (values in `1..n`) | Find All Numbers Disappeared |

**Spot it:** "pair that sums to", "seen before", "duplicate", "group", "anagram", "top k", unsorted input but O(n) wanted.

**Classic bugs:** storing `x` before checking for its partner (it pairs with itself) · using a `list` as a dict key (make it a `tuple`).

**vs. two pointers:** hash map = unsorted, O(n) space. Two pointers = sorted, O(1) space.

## 1.1 Hash map and set lookups

The ladder: plain lookup → canonical key → counting → voting (no map at all) → index marking (the array *is* the map).

### 1.1.1 [1. Two Sum](https://leetcode.com/problems/two-sum/) — ★★★ · Easy

**Problem.** Return indices of the two numbers that add up to `target` (exactly one answer).  
**Example.** `nums = [2, 7, 11, 15], target = 9` → `[0, 1]` (because 2 + 7 = 9)  
**Spot it.** Pair with a given sum, unsorted input, indices required → hash map of value → index.

**Ask first**
- Can I use the same element twice? → No, so look up the partner *before* storing `x`.
- Is the input sorted? → If yes, two pointers does it in O(1) space (that's 167).

> 💡 **Idea.** For each `x` the partner is fixed: `target - x`. Just ask the map "have I seen it?"

**Brute force:** check every pair, O(n²).

**Steps**
1. Keep a map from value to the index where it was seen.
2. For each `x`, compute the partner `target - x`.
3. If the partner is in the map, return its index and the current one.
4. Otherwise store `x` and continue.

**Cost:** Time O(n) (one pass, each lookup is O(1) on average) · Space O(n) (the map can hold every element)

In [2]:
def two_sum(nums, target):
    """Indices i < j with nums[i] + nums[j] == target."""
    # value -> index of everything already scanned
    seen = {}
    for i, x in enumerate(nums):
        # the partner was seen earlier: answer found
        if target - x in seen:
            return [seen[target - x], i]
        # store after the check so x never pairs with itself
        seen[x] = i
    return []


assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 2, 4], 6) == [1, 2]
assert two_sum([3, 3], 6) == [0, 1]
print(two_sum([2, 7, 11, 15], 9))

[0, 1]


### 1.1.2 [217. Contains Duplicate](https://leetcode.com/problems/contains-duplicate/) — ★★★ · Easy

**Problem.** Return `True` if any value appears at least twice.  
**Example.** `[1, 2, 3, 1]` → `True` (1 appears twice)  
**Spot it.** "Seen before?" → set.

**Ask first**
- Is extra memory allowed? → If not, sort in place and compare neighbours: O(n log n), O(1) space.

> 💡 **Idea.** Duplicate = already in the set when you reach it.

**Brute force:** compare every pair, O(n²).

**Steps**
1. Start with an empty set.
2. For each value, if it is in the set, return `True`.
3. Otherwise add it.
4. Finishing the loop means no duplicate.

**Cost:** Time O(n) (one pass of O(1) set operations) · Space O(n) (the set may hold all values)

In [3]:
def contains_duplicate(nums):
    """True if any value repeats."""
    seen = set()
    for x in nums:
        # second sighting: duplicate found, stop early
        if x in seen:
            return True
        seen.add(x)
    return False


assert contains_duplicate([1, 2, 3, 1])
assert not contains_duplicate([1, 2, 3, 4])
print(contains_duplicate([1, 1, 1, 3, 3, 4, 3, 2, 4, 2]))

True


### 1.1.3 [242. Valid Anagram](https://leetcode.com/problems/valid-anagram/) — ★★★ · Easy

**Problem.** Is `t` a rearrangement of `s`?  
**Example.** `s = "listen", t = "silent"` → `True`; `s = "rat", t = "car"` → `False`  
**Spot it.** Same multiset of characters → compare counts.

**Ask first**
- Only `a–z`, or Unicode? → `a–z` fits a 26-slot array; Unicode needs a `Counter`.

> 💡 **Idea.** Anagrams have exactly the same letter counts.

**Brute force:** sort both and compare, O(n log n).

**Steps**
1. If the lengths differ, return `False`.
2. Count the characters of each string.
3. Return whether the two count tables are equal.

**Cost:** Time O(n) (one pass per string) · Space O(1) (at most 26 lowercase keys)

In [4]:
def is_anagram(s, t):
    """True if t is a permutation of s."""
    # different lengths can never match; otherwise compare letter counts
    return len(s) == len(t) and Counter(s) == Counter(t)


assert is_anagram("anagram", "nagaram")
assert not is_anagram("rat", "car")
print(is_anagram("listen", "silent"))

True


### 1.1.4 [49. Group Anagrams](https://leetcode.com/problems/group-anagrams/) — ★★★ · Medium

**Problem.** Group the strings that are anagrams of each other.  
**Example.** `["eat", "tea", "tan", "ate", "nat", "bat"]` → `[["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]`  
**Spot it.** Grouping by an equivalence → hash map from a canonical key to a list.

**Ask first**
- Only lowercase letters? → Yes → a 26-count tuple is the key. With Unicode, use the sorted string as the key (O(k log k) per word).

> 💡 **Idea.** Give every word a *name* that all its anagrams share — its letter counts as a tuple — and bucket words by that name.

**Brute force:** compare each word with every group, O(n²·k).

**Steps**
1. For each word, build its 26-letter count.
2. Turn the count into a tuple so it can be a dict key.
3. Append the word to the list stored under that key.
4. Return all the lists.

**Cost:** Time O(n · k) · Space O(n · k) (every word is stored once)

**Likely follow-ups**
- **Sorted-string key vs count key — which is faster?** Sorting is O(k log k) per word, counting is O(k). For short words sorting is simpler and fine; for long words counting wins.
- **Input is huge and doesn't fit in memory?** Map-reduce it: each worker emits `(key, word)`, the shuffle groups by key, reducers write each group. Same key idea, distributed.
- **Group *shifted* strings instead (abc ~ bcd, LC 249)?** Change the key to the gaps between letters: `tuple((ord(c) - ord(w[0])) % 26 for c in w)`.

In [5]:
def group_anagrams(strs):
    """Buckets of mutually anagrammatic words."""
    groups = defaultdict(list)
    for word in strs:
        # letter-count vector: identical for all anagrams
        count = [0] * 26
        for ch in word:
            count[ord(ch) - ord("a")] += 1
        # tuple makes the vector hashable
        groups[tuple(count)].append(word)
    return list(groups.values())


out = group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
assert sorted(map(sorted, out)) == [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]
print(out)

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


### 1.1.5 [347. Top K Frequent Elements](https://leetcode.com/problems/top-k-frequent-elements/) — ★★★ · Medium

**Problem.** Return the `k` most frequent values.  
**Example.** `nums = [1, 1, 1, 2, 2, 3], k = 2` → `[1, 2]` (1 appears 3 times, 2 appears twice)  
**Spot it.** "Top k" by frequency → count, then bucket sort (or heap).

**Ask first**
- Can there be ties at the k-th spot? → Guaranteed no. If ties are possible, ask which to keep (e.g. smaller value) and use a heap with a tie-break key.

> 💡 **Idea.** A count can't be bigger than `n`, so drop each value into bucket `count` and read buckets from the top — no sorting.

**Brute force:** count, then sort by count, O(n log n).

**Steps**
1. Count each value.
2. Make `n + 1` buckets and put each value in the bucket of its frequency.
3. Walk buckets from high frequency to low, collecting values.
4. Stop as soon as `k` values are collected.

**Cost:** Time O(n) · Space O(n) (counts and buckets)

**Likely follow-ups**
- **Data is a stream (no "final" counts)?** Keep a running `Counter` plus a size-k min-heap keyed by count. Buckets need final counts, the heap doesn't. O(log k) per update.
- **Can you beat O(n log n) without buckets?** Size-k min-heap: O(n log k). Or quickselect on the counts: O(n) average, O(n²) worst — mention random pivots.
- **Way too much data for one machine?** Shard by hash of the value, count each shard, take each shard's top k, merge. (Approximate: Count-Min Sketch + heap.)

In [6]:
def top_k_frequent(nums, k):
    """k values with the highest counts."""
    # buckets[f] holds every value seen exactly f times
    buckets = [[] for _ in range(len(nums) + 1)]
    for value, freq in Counter(nums).items():
        buckets[freq].append(value)
    # read from the most frequent bucket down
    out = []
    for freq in range(len(buckets) - 1, 0, -1):
        out.extend(buckets[freq])
        if len(out) >= k:
            return out[:k]
    return out


assert sorted(top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == [1, 2]
assert top_k_frequent([1], 1) == [1]
print(top_k_frequent([4, 4, 4, 5, 5, 6], 2))

[4, 5]


### 1.1.6 [238. Product of Array Except Self](https://leetcode.com/problems/product-of-array-except-self/) — ★★★ · Medium

**Problem.** `out[i]` = product of every element except `nums[i]`, without division, in O(n).  
**Example.** `[1, 2, 3, 4]` → `[24, 12, 8, 6]` (the first answer is 2·3·4 = 24)  
**Spot it.** "Everything except i" → prefix product × suffix product.

**Ask first**
- Can there be zeros? → Yes, that's why division is banned.
- Does the output array count as extra space? → No, so O(1) extra is the target.

> 💡 **Idea.** Everything except `i` = (everything left of `i`) × (everything right of `i`). Build the left part in one sweep, multiply the right part in on the way back.

**Brute force:** multiply all others for each `i`, O(n²).

**Steps**
1. Left-to-right pass: `out[i]` = product of all elements before `i`.
2. Right-to-left pass with a running suffix product.
3. Multiply each `out[i]` by the suffix, then fold `nums[i]` into the suffix.

**Cost:** Time O(n) (two linear passes) · Space O(1) extra

**Likely follow-ups**
- **Division *is* allowed — how do you handle zeros?** Count zeros. 0 zeros → `total // x`. 1 zero → only that index gets the product of the rest, all others 0. 2+ zeros → all 0.
- **Do it with O(1) extra space.** Store prefix products in `out`, then one backward pass with a single `suffix` variable (that's the solution here).
- **Numbers can overflow?** Use big ints, take results mod a prime if asked, or work in log space (sum of logs) when only comparisons matter.

In [7]:
def product_except_self(nums):
    """Product of all other elements for each index."""
    n = len(nums)
    # out[i] = product of everything left of i
    out = [1] * n
    for i in range(1, n):
        out[i] = out[i - 1] * nums[i - 1]
    # multiply in the product of everything right of i
    suffix = 1
    for i in range(n - 1, -1, -1):
        out[i] *= suffix
        suffix *= nums[i]
    return out


assert product_except_self([1, 2, 3, 4]) == [24, 12, 8, 6]
assert product_except_self([-1, 1, 0, -3, 3]) == [0, 0, 9, 0, 0]
print(product_except_self([1, 2, 3, 4]))

[24, 12, 8, 6]


### 1.1.7 [128. Longest Consecutive Sequence](https://leetcode.com/problems/longest-consecutive-sequence/) — ★★★ · Medium

**Problem.** Length of the longest run of consecutive integers (any order), in O(n).  
**Example.** `[100, 4, 200, 1, 3, 2]` → `4` (the run 1, 2, 3, 4)  
**Spot it.** O(n) forbids sorting → set + only start counting at sequence starts.

**Ask first**
- Duplicates? → Yes, so use a set (a repeat must not extend a run).

> 💡 **Idea.** Only start counting from the *beginning* of a run (`x - 1` not in the set). Then every run is walked once.

**Brute force:** sort and scan, O(n log n) — breaks the O(n) requirement.

**Steps**
1. Put all numbers in a set.
2. For each `x` in the set, skip it if `x - 1` is present (not a run start).
3. From a start, count `x + 1, x + 2, …` while they are in the set.
4. Keep the longest count.

**Cost:** Time O(n) · Space O(n) (the set)

**Likely follow-ups**
- **Why is the nested `while` still O(n)?** The inner walk only starts at run heads, so each number is stepped over at most once in total: O(n + n).
- **Numbers arrive one at a time (streaming)?** Union–find or a map `x → run length` stored at both run ends; when `x` arrives, merge with `x-1`'s and `x+1`'s runs in O(1).
- **Return the sequence, not just the length.** Remember the best start `x` and length `L`; return `range(x, x + L)`.

In [8]:
def longest_consecutive(nums):
    """Length of the longest consecutive-integer run."""
    values = set(nums)
    best = 0
    for x in values:
        # only run heads start a count
        if x - 1 not in values:
            length = 1
            # walk upward while the next integer exists
            while x + length in values:
                length += 1
            best = max(best, length)
    return best


assert longest_consecutive([100, 4, 200, 1, 3, 2]) == 4
assert longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]) == 9
assert longest_consecutive([]) == 0
print(longest_consecutive([100, 4, 200, 1, 3, 2]))

4


### 1.1.8 [271. Encode and Decode Strings](https://leetcode.com/problems/encode-and-decode-strings/) — ★★★ · Medium · Premium

**Problem.** Serialise a list of strings into one string and back; strings may contain any character.  
**Example.** `["hi", "yo"]` → encode to one string → decode → `["hi", "yo"]` again  
**Spot it.** Arbitrary content → length-prefix framing (a delimiter alone can be ambiguous).

**Ask first**
- Can strings contain any character, including my delimiter? → Yes, so plain `join` breaks — use a length prefix.
- Empty strings or an empty list? → Both possible; `[]` and `[""]` must decode differently.

> 💡 **Idea.** Write each string's length in front of it (`5#hello`). The decoder reads the length and grabs exactly that many characters — so any character, even `#`, is safe.

**Brute force:** join with a delimiter → breaks when a string contains it.

**Steps**
1. Encode each string as `len#text` and concatenate.
2. To decode, read digits up to the next `#` to get the length.
3. Take exactly that many characters after `#` as one string.
4. Jump past them and repeat.

**Cost:** Time O(total length) (every character is written and read once) · Space O(total length) (same as time)

**Likely follow-ups**
- **Make it work for any character set (Unicode)?** Length-prefixing doesn't care what the characters are; just count lengths in the same unit you slice in (characters or bytes).
- **Can you do it with escaping instead?** Yes: escape `#` as `##` and use `" # "` as a separator. Works, but decoding is fiddlier — the length header is cleaner.
- **Make the header fixed-width?** Write the length as 4 bytes/digits — no `#` needed; decoder reads 4, then the string.

In [9]:
def encode(strs):
    """Length-prefixed concatenation."""
    # each piece is "<length>#<text>"
    return "".join(f"{len(s)}#{s}" for s in strs)


def decode(data):
    """Inverse of encode."""
    out = []
    i = 0
    while i < len(data):
        # the header ends at the first '#' after i
        j = data.index("#", i)
        length = int(data[i:j])
        # take exactly `length` chars; '#' inside them is harmless
        out.append(data[j + 1 : j + 1 + length])
        i = j + 1 + length
    return out


for case in [["lint", "code", "love", "you"], ["", "a#b", "12#"], []]:
    assert decode(encode(case)) == case
print(encode(["a#b", ""]))

3#a#b0#


### 1.1.9 [36. Valid Sudoku](https://leetcode.com/problems/valid-sudoku/) — ★★ · Medium

**Problem.** Check a partially filled 9×9 board: no repeated digit in any row, column or 3×3 box.  
**Example.** a board where one row has two `5`s → `False`; a board with no repeats → `True`  
**Spot it.** Several "no duplicates within group" constraints → one set per group.

**Ask first**
- Must the board be solvable? → No, only check filled cells. (Solvable would need backtracking — LC 37.)

> 💡 **Idea.** Each filled cell belongs to one row, one column and one box. Keep a set for each of the 27 groups; a repeat anywhere means invalid.

**Brute force:** check the 27 groups one at a time — same O(81), just three passes.

**Steps**
1. Create 9 row sets, 9 column sets, 9 box sets.
2. For every filled cell, compute its box index `(r // 3) * 3 + c // 3`.
3. If the digit is already in its row, column or box set, return `False`.
4. Otherwise add it to all three.

**Cost:** Time O(81) = O(1) (the board size is fixed) · Space O(81) = O(1) (same as time)

**Likely follow-ups**
- **Use less memory?** Replace each set with a 9-bit int: `if mask >> d & 1: invalid; mask |= 1 << d`.
- **Generalise to an N×N board?** Box index becomes `(r // √N) * √N + c // √N`; cost O(N²).
- **Now actually solve it (LC 37)?** Backtracking: try digits in an empty cell, keep row/col/box sets, undo on failure.

In [10]:
def is_valid_sudoku(board):
    """True if no row/column/box repeats a digit."""
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    for r in range(9):
        for c in range(9):
            d = board[r][c]
            if d == ".":
                continue
            # flatten the 3x3 box coordinate into 0..8
            b = (r // 3) * 3 + c // 3
            # any group already holding d breaks the rules
            if d in rows[r] or d in cols[c] or d in boxes[b]:
                return False
            rows[r].add(d)
            cols[c].add(d)
            boxes[b].add(d)
    return True


board = [
    list("53..7...."),
    list("6..195..."),
    list(".98....6."),
    list("8...6...3"),
    list("4..8.3..1"),
    list("7...2...6"),
    list(".6....28."),
    list("...419..5"),
    list("....8..79"),
]
assert is_valid_sudoku(board)
board[0][0] = "8"
assert not is_valid_sudoku(board)
print("ok")

ok


### 1.1.10 [383. Ransom Note](https://leetcode.com/problems/ransom-note/) — ★★ · Easy

**Problem.** Can `ransom_note` be built from the letters of `magazine` (each letter used once)?  
**Example.** `ransom_note = "aa", magazine = "aab"` → `True`; `magazine = "ab"` → `False` (only one `a`)  
**Spot it.** Multiset containment → counts.

**Ask first**
- Can a magazine letter be reused? → No, each is used once — so compare counts, not just presence.

> 💡 **Idea.** Buildable iff the magazine has at least as many of every letter as the note needs.

**Brute force:** search-and-delete each note letter from the magazine, O(m·n).

**Steps**
1. Count the magazine letters.
2. Count the note letters.
3. Subtract: any letter left with a positive count is missing.

**Cost:** Time O(m + n) (one count per string) · Space O(1) (at most 26 keys)

In [ ]:
def can_construct(ransom_note, magazine):
    res = [0] * 26

    # count what the magazine offers
    for x in magazine:
        res[ord(x) - ord('a')] += 1    

    # use one letter up  
    for x in ransom_note:
        res[ord(x) - ord('a')] -= 1
        
        # magazine ran out of this letter      
        if res[ord(x) - ord('a')] < 0:   
            return False
    return True


assert not can_construct("a", "b")
assert not can_construct("aa", "ab")
assert can_construct("aa", "aab")
print(can_construct("aa", "aab"))

True


### 1.1.11 [219. Contains Duplicate II](https://leetcode.com/problems/contains-duplicate-ii/) — ★★ · Easy

**Problem.** Are there `i != j` with `nums[i] == nums[j]` and `|i - j| <= k`?  
**Example.** `nums = [1, 2, 3, 1], k = 3` → `True` (the two 1s are 3 positions apart)  
**Spot it.** Duplicate within distance k → map value → last index (or a size-k window set).

**Ask first**
- Is it `|i − j| ≤ k` or `< k`? → `≤ k` — off-by-one changes the answer.

> 💡 **Idea.** When a value shows up again, only its *most recent* index matters — store that.

**Brute force:** check the next k items for each `i`, O(n·k).

**Steps**
1. Keep a map value → most recent index.
2. For each `i, x`, if `x` was seen and `i - last[x] <= k`, return `True`.
3. Update `last[x] = i`.

**Cost:** Time O(n) (one pass) · Space O(n) (the map (a size-k sliding set gives O(k)))

In [12]:
def contains_nearby_duplicate(nums, k):
    """True if equal values occur within k positions."""
    # value -> most recent index (the closest candidate)
    last = {}
    for i, x in enumerate(nums):
        if x in last and i - last[x] <= k:
            return True
        last[x] = i
    return False


assert contains_nearby_duplicate([1, 2, 3, 1], 3)
assert contains_nearby_duplicate([1, 0, 1, 1], 1)
assert not contains_nearby_duplicate([1, 2, 3, 1, 2, 3], 2)
print("ok")

ok


### 1.1.12 [169. Majority Element](https://leetcode.com/problems/majority-element/) — ★★ · Easy

**Problem.** Return the element that appears more than ⌊n/2⌋ times (guaranteed to exist).  
**Example.** `[2, 2, 1, 1, 1, 2, 2]` → `2` (it appears 4 times out of 7)  
**Spot it.** Strict majority + O(1) space → Boyer–Moore voting.

**Ask first**
- Is a majority guaranteed? → Yes. If not, add a second pass to count the survivor.

> 💡 **Idea.** Pair off each majority copy against a different value. The majority has more copies than everyone else combined, so it survives.

**Brute force:** hash-map counts, O(n) space.

**Steps**
1. Keep a `candidate` and a `count`.
2. When `count` is 0, adopt the current value as the candidate.
3. Same value → `count += 1`; different value → `count -= 1`.
4. The final candidate is the majority.

**Cost:** Time O(n) (one pass) · Space O(1) (two variables)

In [13]:
def majority_element(nums):
    """Value appearing more than n // 2 times."""
    candidate, count = 0, 0
    for x in nums:
        # all votes cancelled: start a new candidate
        if count == 0:
            candidate = x
        # a match adds a vote, a mismatch cancels one
        count += 1 if x == candidate else -1
    return candidate


assert majority_element([3, 2, 3]) == 3
assert majority_element([2, 2, 1, 1, 1, 2, 2]) == 2
print(majority_element([2, 2, 1, 1, 1, 2, 2]))

2


### 1.1.13 [229. Majority Element II](https://leetcode.com/problems/majority-element-ii/) — ★★ · Medium

**Problem.** Return all values appearing more than ⌊n/3⌋ times.  
**Example.** `[3, 2, 3]` → `[3]` (3 appears twice, more than 3/3 = 1 time)  
**Spot it.** Generalised Boyer–Moore: at most k−1 elements can exceed n/k.

**Ask first**
- Could there be no answer? → Yes, so you must recount the two survivors (e.g. `[1,2,3]` has none).

> 💡 **Idea.** At most two values can appear more than n/3 times. Run voting with two candidates — cancelling three different values never wipes out a real answer — then recount to confirm.

**Brute force:** hash-map counts, O(n) space.

**Steps**
1. Keep two candidates with counts.
2. A match increments its count; an empty slot adopts the value.
3. A value matching neither, with both slots full, decrements both (cancels a triple).
4. Recount the survivors and keep those above n/3.

**Cost:** Time O(n) (one voting pass plus one verification pass) · Space O(1)

**Likely follow-ups**
- **Generalise to more than n/k times?** Keep k − 1 candidates and counters (Misra–Gries). O(n·k) time, O(k) space, then verify.
- **Why at most two answers?** Three values each above n/3 would need more than n elements in total.
- **Data is a stream you can read only once?** Misra–Gries still finds every possible heavy hitter in one pass; you need a second pass (or exact counts for just the candidates) to confirm.

In [14]:
def majority_element_ii(nums):
    """All values with count > n // 3."""
    c1, c2, n1, n2 = None, None, 0, 0
    for x in nums:
        # vote for a matching candidate
        if x == c1:
            n1 += 1
        elif x == c2:
            n2 += 1
        # fill an empty slot
        elif n1 == 0:
            c1, n1 = x, 1
        elif n2 == 0:
            c2, n2 = x, 1
        # three distinct values: cancel one of each
        else:
            n1 -= 1
            n2 -= 1
    # survivors are only candidates: verify their real counts
    return [c for c in {c1, c2} if c is not None and nums.count(c) > len(nums) // 3]


assert majority_element_ii([3, 2, 3]) == [3]
assert sorted(majority_element_ii([1, 2])) == [1, 2]
assert majority_element_ii([1]) == [1]
print(majority_element_ii([1, 1, 1, 3, 3, 2, 2, 2]))

[1, 2]


### 1.1.14 [409. Longest Palindrome](https://leetcode.com/problems/longest-palindrome/) — ★★ · Easy

**Problem.** Length of the longest palindrome that can be *built* from the letters of `s`.  
**Example.** `"abccccdd"` → `7` (build `"dccaccd"`)  
**Spot it.** Palindrome from a multiset → pairs plus at most one odd centre.

**Ask first**
- Case-sensitive? → Yes, `A` and `a` are different letters.

> 💡 **Idea.** A palindrome is mirrored pairs plus at most one odd letter in the middle.

**Brute force:** try arrangements, factorial time.

**Steps**
1. Count each letter.
2. Add `count // 2 * 2` for every letter (all usable pairs).
3. If any count is odd, add 1 for the centre.

**Cost:** Time O(n) (one count) · Space O(1) (bounded alphabet)

In [15]:
def longest_palindrome(s):
    counts = Counter(s)
    length = 0
    has_odd = False
    for c in counts.values():
        # use as many pairs as possible
        length += c // 2 * 2
        # remember if any letter has one left over
        if c % 2 == 1:
            has_odd = True
    # one leftover letter can sit in the middle
    if has_odd:
        length += 1
    return length


assert longest_palindrome("abccccdd") == 7
assert longest_palindrome("a") == 1
print(longest_palindrome("abccccdd"))

7


### 1.1.15 [12. Integer to Roman](https://leetcode.com/problems/integer-to-roman/) — ★★ · Medium

**Problem.** Convert an integer (1–3999) to a Roman numeral.  
**Example.** `58` → `"LVIII"`; `3` → `"III"`  
**Spot it.** Greedy over a fixed denomination table that already includes the subtractive pairs.

**Ask first**
- Range? → 1–3999, so `M` repeats at most 3 times and the table never runs out.

> 💡 **Idea.** Put the awkward ones (900, 400, 90, 40, 9, 4) in the table as their own symbols, then just greedily take the biggest value that fits.

**Brute force:** per-digit if/else for 4 and 9 — long and easy to get wrong.

**Steps**
1. List the 13 values from 1000 down to 1, including the subtractive pairs.
2. For each value, take as many copies as fit (`num // value`).
3. Append that many symbols and continue with the remainder.

**Cost:** Time O(1) · Space O(1) (same as time)

**Likely follow-ups**
- **Now go back: Roman → Integer (LC 13)?** Scan left to right; if a symbol is smaller than the next one, subtract it, otherwise add it.
- **Validate a Roman numeral?** Easiest: convert to int and back; valid iff you get the same string. Or check rules (max 3 repeats; only I/X/C subtract).
- **Numbers above 3999?** Ask what notation they want (e.g. a bar = ×1000) and add those symbols to the table.

In [16]:
ROMAN_TABLE = [
    (1000, "M"),
    (900, "CM"),
    (500, "D"),
    (400, "CD"),
    (100, "C"),
    (90, "XC"),
    (50, "L"),
    (40, "XL"),
    (10, "X"),
    (9, "IX"),
    (5, "V"),
    (4, "IV"),
    (1, "I"),
]


def int_to_roman(num):
    """Roman numeral for num."""
    out = []
    for value, symbol in ROMAN_TABLE:
        # how many times this value fits
        count = num // value
        out.append(symbol * count)
        # what is left
        num = num % value
    return "".join(out)


assert int_to_roman(3749) == "MMMDCCXLIX"
assert int_to_roman(58) == "LVIII"
assert int_to_roman(1994) == "MCMXCIV"
print(int_to_roman(1994))

MCMXCIV


### 1.1.16 [274. H-Index](https://leetcode.com/problems/h-index/) — ★★ · Medium

**Problem.** Largest `h` such that the researcher has at least `h` papers with ≥ `h` citations each.  
**Example.** `[3, 0, 6, 1, 5]` → `3` (3 papers have at least 3 citations)  
**Spot it.** Threshold on counts bounded by n → counting sort (or sort descending).

**Ask first**
- Is the input sorted? → If yes, binary search does it in O(log n) (that's H-Index II, LC 275).

> 💡 **Idea.** `h` can't be more than `n`, so cap every citation at `n` and count papers per bucket. Walk down from `n` until the running count reaches `h`.

**Brute force:** sort descending, find the last `i` with `citations[i] ≥ i + 1`, O(n log n).

**Steps**
1. Bucket each citation count into `min(c, n)`.
2. Walk `h` from n down to 0, accumulating how many papers have ≥ h citations.
3. Return the first `h` where that running count reaches `h`.

**Cost:** Time O(n) (one bucketing pass and one walk over n + 1 buckets) · Space O(n) (the buckets)

**Likely follow-ups**
- **Citations come sorted ascending (LC 275)?** Binary search for the first `i` with `citations[i] ≥ n − i`; answer `n − i`. O(log n).
- **Papers and citations arrive as a stream?** Keep a min-heap of the top `h` citation counts; when a new count beats the smallest and `h` can grow, push and raise `h`.

In [ ]:
def h_index(citations):
    """Researcher's h-index."""
    n = len(citations)
    # counts above n act like n, so n + 1 buckets suffice
    buckets = [0] * (n + 1)
    for c in citations: 
        buckets[min(c, n)] += 1
    # papers with at least h citations, accumulated from the top
    at_least = 0
    for h in range(n, -1, -1):
        at_least += buckets[h]
        if at_least >= h:
            return h
    return 0


assert h_index([3, 0, 6, 1, 5]) == 3
assert h_index([1, 3, 1]) == 1
assert h_index([0]) == 0
print(h_index([3, 0, 6, 1, 5]))

3


### 1.1.17 [448. Find All Numbers Disappeared in an Array](https://leetcode.com/problems/find-all-numbers-disappeared-in-an-array/) — ★★ · Easy

**Problem.** Values are in `1..n`; return the ones that never appear, with O(1) extra space.  
**Example.** `[4, 3, 2, 7, 8, 2, 3, 1]` → `[5, 6]`  
**Spot it.** Values index the array → mark presence by negating `nums[v-1]`.

**Ask first**
- Can I modify the input? → If not, use a set (O(n) space).

> 💡 **Idea.** Values are `1..n`, so value `v` can mark slot `v − 1` — flip its sign to say "v seen". Slots still positive are missing.

**Brute force:** a set of seen values, O(n) space.

**Steps**
1. For each value, take `abs(v)` (it may already be negated).
2. Make `nums[abs(v) - 1]` negative.
3. Indices still holding positive numbers are the missing values (index + 1).

**Cost:** Time O(n) (two passes) · Space O(1) extra (the output list does not count)

In [18]:
def find_disappeared_numbers(nums):
    """Values in 1..n absent from nums."""
    for v in nums:
        # abs: this slot may already have been negated
        idx = abs(v) - 1
        # negative sign at idx records "idx + 1 was seen"
        nums[idx] = -abs(nums[idx])
    # still positive means that value never appeared
    return [i + 1 for i, v in enumerate(nums) if v > 0]


assert find_disappeared_numbers([4, 3, 2, 7, 8, 2, 3, 1]) == [5, 6]
assert find_disappeared_numbers([1, 1]) == [2]
print(find_disappeared_numbers([4, 3, 2, 7, 8, 2, 3, 1]))

[5, 6]


### 1.1.18 [454. 4Sum II](https://leetcode.com/problems/4sum-ii/) — ★ · Medium

**Problem.** Count tuples `(i, j, k, l)` with `a[i] + b[j] + c[k] + d[l] == 0` (four arrays of length n).  
**Example.** `a = [1, 2], b = [-2, -1], c = [-1, 2], d = [0, 2]` → `2`  
**Spot it.** Meet in the middle — split four loops into two halves of two.

**Ask first**
- Count index tuples or distinct value tuples? → Index tuples, so duplicate values count separately — a count map handles that.

> 💡 **Idea.** `a + b + c + d = 0` ⇔ `a + b = −(c + d)`. Count all `a + b` sums, then look up `−(c + d)` for every pair. Two Sum where each "number" is a pair sum.

**Brute force:** four nested loops, O(n⁴).

**Steps**
1. Count every pair sum `a[i] + b[j]` in a hash map.
2. For every pair `c[k], d[l]`, add the count of `-(c[k] + d[l])`.
3. Return the total.

**Cost:** Time O(n²) (two double loops) · Space O(n²) (up to n² distinct pair sums)

**Likely follow-ups**
- **Generalise to k arrays?** Split them into two halves, build a count map of sums for the first half, look up negated sums from the second: O(n^(k/2)).
- **Return the tuples, not the count?** Store lists of `(i, j)` per sum instead of counts — output size can be O(n⁴), so warn about that.
- **Memory is tight?** Sort the two pair-sum lists and two-pointer them from opposite ends: O(n² log n) time, still O(n²) space but no hash map.

In [19]:
def four_sum_count(a, b, c, d):
    """Number of zero-sum quadruples across four arrays."""
    # first half: how often each a + b sum occurs
    ab = Counter(x + y for x in a for y in b)
    # second half: each c + d needs a partner of -(c + d)
    return sum(ab[-(x + y)] for x in c for y in d)


assert four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]) == 2
assert four_sum_count([0], [0], [0], [0]) == 1
print(four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]))

2


## 1.2 In-place array manipulation

No extra memory allowed? Use a **write pointer**: one index reads everything, another marks where the next keeper goes.

> **Think of it like…** packing a suitcase from a pile — your hand picks up each item, and only keepers go into the next free spot.

**Classic bugs:** filling a merge from the front (overwrites unread data) · advancing `mid` after swapping in an unseen value (Sort Colors) · forgetting `k %= n` before rotating.

### 1.2.1 [88. Merge Sorted Array](https://leetcode.com/problems/merge-sorted-array/) — ★★ · Easy

**Problem.** Merge sorted `nums2` into sorted `nums1`, which has `n` spare slots at the end.  
**Example.** `nums1 = [1, 2, 3, 0, 0, 0], m = 3, nums2 = [2, 5, 6], n = 3` → `nums1 = [1, 2, 2, 3, 5, 6]`  
**Spot it.** In-place merge with free space at the back → fill from the back.

**Ask first**
- Can `m` or `n` be 0? → Yes; the loop must still copy any leftover `nums2`.

> 💡 **Idea.** The free space is at the back of `nums1`, so fill from the back with the biggest remaining element — you never overwrite something you still need.

**Brute force:** paste `nums2` at the end and sort, O((m+n) log(m+n)).

**Steps**
1. Point `i` at the last real element of `nums1`, `j` at the last of `nums2`, `k` at the last slot.
2. Write the larger of `nums1[i]` and `nums2[j]` at `k` and move that pointer left.
3. Move `k` left after every write.
4. When `nums2` is exhausted, stop — the rest of `nums1` is already in place.

**Cost:** Time O(m + n) (each element is written once) · Space O(1)

In [20]:
def merge(nums1, m, nums2, n):
    """Merge nums2 into nums1 in place."""
    # i, j: last unread of each list; k: last free slot
    i, j, k = m - 1, n - 1, m + n - 1
    while j >= 0:
        # write the larger tail element into the back slot
        if i >= 0 and nums1[i] > nums2[j]:
            nums1[k] = nums1[i]
            i -= 1
        else:
            nums1[k] = nums2[j]
            j -= 1
        k -= 1


a = [1, 2, 3, 0, 0, 0]
merge(a, 3, [2, 5, 6], 3)
assert a == [1, 2, 2, 3, 5, 6]
b = [0]
merge(b, 0, [1], 1)
assert b == [1]
print(a)

[1, 2, 2, 3, 5, 6]


### 1.2.2 [26. Remove Duplicates from Sorted Array](https://leetcode.com/problems/remove-duplicates-from-sorted-array/) — ★★ · Easy

**Problem.** Remove duplicates in place from a sorted array; return the number of unique values.  
**Example.** `[1, 1, 2]` → returns `2`, and `nums` starts with `[1, 2]`  
**Spot it.** Sorted → duplicates are adjacent → compare with last written value.

**Ask first**
- Is it sorted? → Yes; unsorted would need a set.

> 💡 **Idea.** Sorted → duplicates sit next to each other. Keep `x` only if it differs from the last value you wrote.

**Brute force:** build a new list of unique values, O(n) space.

**Steps**
1. Start `write = 0`.
2. Keep `x` if `write == 0` or `x != nums[write - 1]`.
3. Copy and advance on keep; return `write`.

**Cost:** Time O(n) · Space O(1)

In [21]:
def remove_duplicates(nums):
    """Unique-compact a sorted array in place."""
    write = 0
    for x in nums:
        # new value iff it differs from the last kept one
        if write == 0 or x != nums[write - 1]:
            nums[write] = x
            write += 1
    return write


a = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]
k = remove_duplicates(a)
assert a[:k] == [0, 1, 2, 3, 4]
print(k)

5


### 1.2.3 [80. Remove Duplicates from Sorted Array II](https://leetcode.com/problems/remove-duplicates-from-sorted-array-ii/) — ★★ · Medium

**Problem.** Same as above but keep each value at most twice.  
**Example.** `[1, 1, 1, 2, 2, 3]` → returns `5`, and `nums` starts with `[1, 1, 2, 2, 3]`  
**Spot it.** Generalises: allow `x` if it differs from the element two slots back in the output.

**Ask first**
- At most 2 copies, or k? → 2 here; the same code works for k with `nums[write - k]`.

> 💡 **Idea.** Keep `x` unless it equals the value two slots back in the output — then two copies are already kept.

**Brute force:** track a run counter and reset on each new value — works, just more state.

**Steps**
1. Start `write = 0`.
2. Keep `x` if `write < 2` or `x != nums[write - 2]`.
3. Copy and advance on keep; return `write`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Allow at most k copies?** Keep `x` if `write < k or x != nums[write - k]`. Same O(n), O(1).
- **The array isn't sorted?** Use a count map and keep `x` while `count[x] < k` — O(n) time but O(n) space, and order is the original order.

In [22]:
def remove_duplicates_ii(nums):
    """Keep at most two copies of each value in a sorted array."""
    write = 0
    for x in nums:
        # equal to the value two slots back means two copies already kept
        if write < 2 or x != nums[write - 2]:
            nums[write] = x
            write += 1
    return write


a = [0, 0, 1, 1, 1, 1, 2, 3, 3]
k = remove_duplicates_ii(a)
assert a[:k] == [0, 0, 1, 1, 2, 3, 3]
print(k)

7


### 1.2.4 [283. Move Zeroes](https://leetcode.com/problems/move-zeroes/) — ★★ · Easy

**Problem.** Move all zeros to the end, keeping the order of non-zeros, in place.  
**Example.** `[0, 1, 0, 3, 12]` → `[1, 3, 12, 0, 0]`  
**Spot it.** Stable partition → write pointer with swaps.

**Ask first**
- Must non-zero order stay the same? → Yes, so swap forward in order (no two-ends trick).

> 💡 **Idea.** Swap each non-zero into the write slot; zeros get pushed behind it automatically.

**Brute force:** copy non-zeros to a new list and pad with zeros, O(n) space.

**Steps**
1. Start `write = 0`.
2. For each non-zero at `i`, swap `nums[i]` with `nums[write]` and advance `write`.
3. Zeros collect after `write`.

**Cost:** Time O(n) (one pass) · Space O(1)

In [23]:
def move_zeroes(nums):
    """Stable move of zeros to the end."""
    write = 0
    for i, x in enumerate(nums):
        # swap the non-zero forward; a zero (if any) drops back to i
        if x != 0:
            nums[write], nums[i] = nums[i], nums[write]
            write += 1


a = [0, 1, 0, 3, 12]
move_zeroes(a)
assert a == [1, 3, 12, 0, 0]
print(a)

[1, 3, 12, 0, 0]


### 1.2.5 [189. Rotate Array](https://leetcode.com/problems/rotate-array/) — ★★ · Medium

**Problem.** Rotate right by `k` steps in place.  
**Example.** `nums = [1, 2, 3, 4, 5, 6, 7], k = 3` → `[5, 6, 7, 1, 2, 3, 4]`  
**Spot it.** Rotation in place → three reversals.

**Ask first**
- Can `k` be bigger than `n`? → Yes, so do `k %= n` first.
- Is O(1) extra space required? → If yes, you need the three-reverse trick; otherwise slicing is fine.

> 💡 **Idea.** Reverse everything (the last k jump to the front, but backwards), then reverse each of the two blocks back into order.

**Brute force:** rotate by one, k times, O(n·k).

**Steps**
1. Reduce `k %= n`.
2. Reverse the whole array.
3. Reverse the first `k` elements.
4. Reverse the remaining `n − k` elements.

**Cost:** Time O(n) (each element is swapped twice) · Space O(1)

**Likely follow-ups**
- **Give three different solutions.** (1) Extra array: `out[(i + k) % n] = nums[i]`, O(n) space. (2) Cyclic replacements: follow each cycle moving items to their target, O(1) space. (3) Three reversals, O(1) space.
- **Rotate left instead?** Left by k = right by `n − k`; or reverse the first k, the rest, then the whole thing.
- **Rotate a linked list (LC 61)?** Find the length, make it a ring, break it `n − k % n` steps from the head.

In [24]:
def rotate(nums, k):
    """Rotate nums right by k in place."""

    def reverse(lo, hi):
        # swap ends inward
        while lo < hi:
            nums[lo], nums[hi] = nums[hi], nums[lo]
            lo, hi = lo + 1, hi - 1

    n = len(nums)
    # rotating by n is a no-op
    k %= n
    # whole reverse brings the last k to the front, backwards
    reverse(0, n - 1)
    # fix the order inside each block
    reverse(0, k - 1)
    reverse(k, n - 1)


a = [1, 2, 3, 4, 5, 6, 7]
rotate(a, 3)
assert a == [5, 6, 7, 1, 2, 3, 4]
b = [-1, -100, 3, 99]
rotate(b, 2)
assert b == [3, 99, -1, -100]
print(a)

[5, 6, 7, 1, 2, 3, 4]


### 1.2.6 [75. Sort Colors](https://leetcode.com/problems/sort-colors/) — ★★ · Medium

**Problem.** Sort an array of 0s, 1s and 2s in one pass, in place.  
**Example.** `[2, 0, 2, 1, 1, 0]` → `[0, 0, 1, 1, 2, 2]`  
**Spot it.** Three-way partition → Dutch national flag (three pointers).

**Ask first**
- Must it be one pass? → If two passes is OK, just count 0/1/2 and rewrite — much simpler.

> 💡 **Idea.** Keep three zones: 0s on the left, 2s on the right, 1s in the middle. `mid` scans the unknown part and throws each value into its zone.

**Brute force:** count, then rewrite (two passes). Any comparison sort is O(n log n).

**Steps**
1. Start `low = mid = 0`, `high = n − 1`.
2. `nums[mid] == 0`: swap to `low`, advance `low` and `mid`.
3. `nums[mid] == 1`: advance `mid`.
4. `nums[mid] == 2`: swap to `high`, shrink `high`, and do *not* advance `mid`.
5. Stop when `mid > high`.

**Cost:** Time O(n) (each step shrinks the unknown region) · Space O(1)

**Likely follow-ups**
- **Sort k colors (k > 3)?** Counting sort: O(n + k) time, O(k) space. With O(1) space: partition by min/max colour each round (O(n·k/2)), or recursive partition by colour range (O(n log k)).
- **Why don't you move `mid` after swapping with `high`?** The value that came from `high` hasn't been looked at yet — you must check it.
- **Objects with a colour key, keep them stable?** Dutch flag isn't stable. Use counting sort with prefix-sum positions (stable, O(n) extra).

In [25]:
def sort_colors(nums):
    """In-place one-pass sort of 0/1/2 values."""
    # [0, low) = 0s, [low, mid) = 1s, (high, end] = 2s, [mid, high] unknown
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        if nums[mid] == 0:
            # grow the 0 region; the swapped-in value is a known 1
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1
            mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            # grow the 2 region; the swapped-in value is unseen, so mid stays
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1


a = [2, 0, 2, 1, 1, 0]
sort_colors(a)
assert a == [0, 0, 1, 1, 2, 2]
print(a)

[0, 0, 1, 1, 2, 2]


### 1.2.7 [912. Sort an Array](https://leetcode.com/problems/sort-an-array/) — ★★ · Medium

**Problem.** Sort an integer array in ascending order in O(n log n) without any built-in sort.  
**Example.** `[5, 2, 3, 1]` → `[1, 2, 3, 5]`  
**Spot it.** "No built-in sort" + O(n log n) required → write merge sort (always O(n log n)).

**Ask first**
- Built-in `sort` allowed? → No (that's the whole point).
- Must it be stable or in place? → Stable → merge sort. In place with a hard worst case → heap sort. Otherwise quicksort with a random pivot.

> 💡 **Idea.** Merge sort: split in half, sort each half, merge two sorted lists. Always O(n log n) — no bad inputs.

**Brute force:** bubble/insertion sort, O(n²) — times out at n = 5·10⁴.

**Steps**
1. A list of 0 or 1 values is already sorted.
2. Sort the left half and the right half recursively.
3. Merge: repeatedly take the smaller front value, then append whatever is left.

**Cost:** Time O(n log n) (log n levels of splitting, O(n) merge work per level) · Space O(n) (the merged lists)

**Likely follow-ups**
- **Why not plain quicksort?** Sorted or all-equal input makes it O(n²). Fix: random pivot or median-of-three, and 3-way partition for duplicates.
- **What do real libraries use?** Python/Java objects: Timsort (merge + insertion, stable, fast on runs). C++ `std::sort`: introsort (quicksort that falls back to heapsort).
- **Data doesn't fit in memory?** External merge sort: sort chunks that fit, write them out, then k-way merge with a min-heap.
- **Values are small integers?** Counting sort, O(n + range).

In [ ]:
def sort_array(nums):
    """Merge sort, ascending."""
    # zero or one value is already sorted
    if len(nums) <= 1:
        return nums
    # sort each half on its own
    mid = len(nums) // 2
    left = sort_array(nums[:mid])
    right = sort_array(nums[mid:])
    
    # merge: always take the smaller front value
    merged, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    # one side ran out; the rest of the other side is already sorted
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


assert sort_array([5, 2, 3, 1]) == [1, 2, 3, 5]
assert sort_array([5, 1, 1, 2, 0, 0]) == [0, 0, 1, 1, 2, 5]
assert sort_array([]) == []
values = [random.randint(-50, 50) for _ in range(200)]
assert sort_array(values) == sorted(values)
print(sort_array([5, 2, 3, 1]))

[1, 2, 3, 5]


### 1.2.8 [31. Next Permutation](https://leetcode.com/problems/next-permutation/) — ★★ · Medium

**Problem.** Rearrange into the next lexicographically larger permutation (wrap to smallest), in place.  
**Example.** `[1, 2, 3]` → `[1, 3, 2]`; `[3, 2, 1]` → `[1, 2, 3]` (wraps around)  
**Spot it.** Classic three-step algorithm: find pivot, swap with successor, reverse suffix.

**Ask first**
- Already the largest permutation? → Wrap around to the smallest (just reverse everything).

> 💡 **Idea.** From the right, find the first dip `nums[i] < nums[i+1]`. Swap `nums[i]` with the smallest bigger number to its right, then reverse the tail (it was descending) to make it as small as possible.

**Brute force:** list all permutations in order and take the next one, O(n!).

**Steps**
1. From the right, find the first `i` with `nums[i] < nums[i + 1]` (the pivot).
2. If there is a pivot, find the rightmost `j` with `nums[j] > nums[i]` and swap them.
3. Reverse the suffix after `i` so it becomes ascending (smallest).
4. No pivot means the array was the last permutation: reversing everything wraps to the first.

**Cost:** Time O(n) (at most three linear scans) · Space O(1)

**Likely follow-ups**
- **Previous permutation instead?** Mirror it: find the first `nums[i] > nums[i+1]` from the right, swap with the largest smaller value on its right, reverse the tail.
- **Get the k-th permutation directly (LC 60)?** Use factorials: the first digit is `digits[(k−1) // (n−1)!]`, remove it, repeat with the remainder. O(n²).
- **Next greater number with the same digits (LC 556)?** Same algorithm on the digit list; return -1 if none or if it overflows 32 bits.

In [ ]:
def next_permutation(nums):
    """Advance nums to its next permutation in place."""
    # pivot: first i from the right with nums[i] < nums[i + 1]
    i = len(nums) - 2
    
    while i >= 0 and nums[i] >= nums[i + 1]:
        i -= 1

    if i >= 0:
        # rightmost value larger than the pivot is its smallest successor
        j = len(nums) - 1
        while nums[j] <= nums[i]:
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]
    # the suffix is descending: reverse it to its smallest order
    nums[i + 1 :] = reversed(nums[i + 1 :])


for before, after in [
    ([1, 2, 3], [1, 3, 2]),
    ([3, 2, 1], [1, 2, 3]),
    ([1, 1, 5], [1, 5, 1]),
]:
    next_permutation(before)
    assert before == after
a = [1, 3, 5, 4, 2]
next_permutation(a)
print(a)

[1, 4, 2, 3, 5]


---
# 2) Two Pointers

**The gist.** Two indices walk the data, and every step rules out one element for good — so a double loop becomes a single pass.

> **Think of it like…** two people at either end of a sorted bookshelf looking for two books that add up to 500 pages. Too few pages? The left person steps right. Too many? The right person steps left. Nobody walks back.

| Shape | Pointers | Why skipping is safe |
|---|---|---|
| Opposite ends | `lo = 0`, `hi = n-1`, move inward | sorted order proves one end can't be in a better answer |
| Same direction | `slow`, `fast` | `slow` marks the finished part, `fast` reads ahead |

**Spot it:** sorted + pair/triplet sum · palindrome · in place / O(1) space · best pair `(i, j)` where moving the worse side is safe.

**Classic bugs:** `lo < hi` (pairs) vs `lo <= hi` (one person can ride alone) · deduping only one level in 3Sum · forgetting to sort first.

**vs. sliding window:** a window also moves two pointers the same way, but the *contents* of the window are the answer.

## 2.1 Opposite-end pointers

Same loop every time — `while lo < hi:` — only the rule for *which pointer moves* changes. Say that rule (and why it's safe) before you code.

### 2.1.1 [125. Valid Palindrome](https://leetcode.com/problems/valid-palindrome/) — ★★★ · Easy

**Problem.** After lower-casing and removing non-alphanumerics, is `s` a palindrome?  
**Example.** `"A man, a plan, a canal: Panama"` → `True`; `"race a car"` → `False`  
**Spot it.** Palindrome check → pointers from both ends.

**Ask first**
- What counts? → Only letters and digits, case-insensitive — this decides your skip rule.

> 💡 **Idea.** Walk in from both ends, skip anything that isn't a letter or digit, compare lowercased.

**Brute force:** build a cleaned, reversed copy and compare, O(n) space.

**Steps**
1. Put `lo` at the start and `hi` at the end.
2. Skip non-alphanumeric characters on either side.
3. Compare the lower-cased characters; a mismatch means `False`.
4. Move both inward until they meet.

**Cost:** Time O(n) (each index moves at most n times in total) · Space O(1)

In [28]:
def is_palindrome(s):
    """Alphanumeric, case-insensitive palindrome test."""
    lo, hi = 0, len(s) - 1
    while lo < hi:
        # skip punctuation and spaces on either side
        if not s[lo].isalnum():
            lo += 1
        elif not s[hi].isalnum():
            hi -= 1
        # mirrored characters must match
        elif s[lo].lower() != s[hi].lower():
            return False
        else:
            lo, hi = lo + 1, hi - 1 
    return True


assert is_palindrome("A man, a plan, a canal: Panama")
assert not is_palindrome("race a car")
assert is_palindrome(" ")
print(is_palindrome("A man, a plan, a canal: Panama"))

True


### 2.1.2 [167. Two Sum II - Input Array Is Sorted](https://leetcode.com/problems/two-sum-ii-input-array-is-sorted/) — ★★ · Medium

**Problem.** Sorted array; return 1-based indices of two numbers summing to `target`, O(1) space.  
**Example.** `numbers = [2, 7, 11, 15], target = 9` → `[1, 2]` (1-based positions of 2 and 7)  
**Spot it.** Sorted + pair sum → opposite pointers.

**Ask first**
- 1-based indices? → Yes, add 1 to both.

> 💡 **Idea.** Sum too small? The left number is too small for *any* partner — move left. Too big? Move right. Each step throws one number away for good.

**Brute force:** hash map like Two Sum — O(n) space, ignores the sorting.

**Steps**
1. `lo` at the start, `hi` at the end.
2. Sum equals target → return.
3. Sum too small → `lo += 1`; too large → `hi -= 1`.
4. Repeat while `lo < hi`.

**Cost:** Time O(n) (every step removes one element from consideration) · Space O(1)

**Likely follow-ups**
- **Count all pairs (with duplicates) that hit the target?** Same scan; on a match with `nums[lo] == nums[hi]` add C(len, 2) and stop, otherwise multiply the run lengths of the two values and skip both runs.
- **Count pairs with sum < target?** When `nums[lo] + nums[hi] < target`, all of `lo+1..hi` work with `lo`: add `hi − lo`, then move `lo`.
- **Input isn't sorted?** Hash map (O(n), O(n) space) or sort first (O(n log n)) — but sorting loses original indices unless you sort `(value, index)` pairs.

In [29]:
def two_sum_sorted(numbers, target):
    """1-based indices of the pair summing to target."""
    lo, hi = 0, len(numbers) - 1
    while lo < hi:
        total = numbers[lo] + numbers[hi]
        if total == target:
            return [lo + 1, hi + 1]
        # too small: numbers[lo] fails even with the largest partner
        if total < target:
            lo += 1
        # too large: numbers[hi] fails even with the smallest partner
        else:
            hi -= 1
    return []


assert two_sum_sorted([2, 7, 11, 15], 9) == [1, 2]
assert two_sum_sorted([2, 3, 4], 6) == [1, 3]
assert two_sum_sorted([-1, 0], -1) == [1, 2]
print(two_sum_sorted([2, 7, 11, 15], 9))

[1, 2]


### 2.1.3 [15. 3Sum](https://leetcode.com/problems/3sum/) — ★★★ · Medium

**Problem.** All unique triplets summing to 0.  
**Example.** `[-1, 0, 1, 2, -1, -4]` → `[[-1, -1, 2], [-1, 0, 1]]`  
**Spot it.** Triplet sum → sort, fix one, two-pointer the rest.

**Ask first**
- Unique triplets by value? → Yes, so you must skip duplicates.

> 💡 **Idea.** Sort. Fix one number, then Two Sum II on the rest with two pointers. Skip repeated values at both levels so no triplet appears twice.

**Brute force:** three loops + a set of sorted triplets, O(n³).

**Steps**
1. Sort the array.
2. For each `i`, skip it if it equals the previous value; stop once `nums[i] > 0`.
3. Run two pointers on `i+1 .. n−1` with target `-nums[i]`.
4. On a hit, record it, move both pointers, and skip equal values on the left.

**Cost:** Time O(n²) · Space O(1) extra (besides the output and the sort)

**Likely follow-ups**
- **3Sum Closest (LC 16)?** Same loop; track the sum with the smallest `abs(sum − target)` and move pointers by comparing to target. O(n²).
- **Count triplets with sum < target (3Sum Smaller)?** On `sum < target`, add `hi − lo` (all those `hi`s work), then `lo += 1`. O(n²).
- **Can't sort / must keep indices?** For each `i`, run a hash-set Two Sum on the rest: O(n²) time, O(n) space; dedupe with a set of sorted tuples.
- **k-Sum in general?** Recurse: fix one, solve (k−1)-Sum, base case Two Sum II. O(n^(k−1)).

In [ ]:
def three_sum(nums):
    """Unique triplets with zero sum."""
    nums = sorted(nums)
    out = []

    for i in range(len(nums) - 2):
        # smallest element positive: no zero sum possible
        if nums[i] > 0:
            break
        
        # same first value gives the same triplets
        if i > 0 and nums[i] == nums[i - 1]:
            continue

        # Two Sum II on the suffix
        lo, hi = i + 1, len(nums) - 1
        while lo < hi:
            total = nums[i] + nums[lo] + nums[hi]
            if total < 0:
                lo += 1
            elif total > 0:
                hi -= 1
            else:
                out.append([nums[i], nums[lo], nums[hi]])
                lo += 1
                hi -= 1
                # skip repeated second values
                while lo < hi and nums[lo] == nums[lo - 1]:
                    lo += 1
    return out


assert three_sum([-1, 0, 1, 2, -1, -4]) == [[-1, -1, 2], [-1, 0, 1]]
assert three_sum([0, 1, 1]) == []
assert three_sum([0, 0, 0, 0]) == [[0, 0, 0]]
print(three_sum([-1, 0, 1, 2, -1, -4]))

[[-1, -1, 2], [-1, 0, 1]]


### 2.1.4 [11. Container With Most Water](https://leetcode.com/problems/container-with-most-water/) — ★★★ · Medium

**Problem.** Choose two lines maximising `min(h[i], h[j]) * (j - i)`.  
**Example.** `[1, 8, 6, 2, 5, 4, 8, 3, 7]` → `49` (the 8 and the 7, 7 apart: 7 × 7)  
**Spot it.** Pair maximisation on an array → opposite pointers, move the limiting side.

**Ask first**
- Width = `j − i`? → Yes, lines have no thickness.

> 💡 **Idea.** Area is limited by the shorter line. Moving the taller one inward can only shrink the width and never raise the height — so always move the shorter one.

**Brute force:** try every pair, O(n²).

**Steps**
1. Start with the widest pair `lo = 0`, `hi = n − 1`.
2. Record the area `min(h[lo], h[hi]) * (hi − lo)`.
3. Move the pointer at the shorter line inward.
4. Repeat until they meet; return the best area.

**Cost:** Time O(n) (one inward pass) · Space O(1)

**Likely follow-ups**
- **Prove moving the shorter line is safe.** Every pair using the shorter line and something further in has smaller width and height ≤ that short line — none can beat the current area, so drop the short line.
- **Water trapped between *all* bars (LC 42)?** Different problem: each bar holds `min(maxLeft, maxRight) − h`. Two pointers again, moving the side with the smaller max.
- **Return the pair of indices too?** Track `(i, j)` when you update the best area.

In [31]:
def max_area(height):
    """Largest water area between two lines."""
    lo, hi, best = 0, len(height) - 1, 0
    while lo < hi:
        # the shorter line sets the water level
        best = max(best, min(height[lo], height[hi]) * (hi - lo))
        # the shorter line can never do better with a narrower partner
        if height[lo] < height[hi]:
            lo += 1
        else:
            hi -= 1
    return best


assert max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]) == 49
assert max_area([1, 1]) == 1
print(max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]))

49


### 2.1.5 [42. Trapping Rain Water](https://leetcode.com/problems/trapping-rain-water/) — ★★ · Hard

**Problem.** Total water trapped between bars of the given heights.  
**Example.** `[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]` → `6`  
**Spot it.** Water at i = `min(max_left, max_right) − h[i]` → two pointers with running maxima.

**Ask first**
- Bars of width 1, heights ≥ 0? → Yes; fewer than 3 bars hold nothing.

> 💡 **Idea.** Water above a bar = `min(tallest on its left, tallest on its right) − its height`. With two pointers, the side with the smaller max is already decided — settle it and move in.

**Brute force:** for each bar scan left and right for maxes, O(n²).

**Steps**
1. Pointers at both ends; `left_max = right_max = 0`.
2. Move the side with the lower bar (its max is the bottleneck).
3. Update that side's max, add `side_max − height` to the water.
4. Repeat until the pointers meet.

**Cost:** Time O(n) (each bar is settled once) · Space O(1)

**Likely follow-ups**
- **2D version — water on a height map (LC 407)?** Min-heap from the border inward (like Dijkstra): pop the lowest wall, water a neighbour up to that level, push it.
- **Do it with a stack?** Monotonic decreasing stack; when a taller bar comes, pop and add water layer by layer. O(n).

In [32]:
def trap(height):
    """Units of rain water trapped."""
    lo, hi = 0, len(height) - 1
    left_max = right_max = water = 0
    while lo < hi:
        # left side is lower: its own max bounds the water at lo
        if height[lo] < height[hi]:
            left_max = max(left_max, height[lo])
            water += left_max - height[lo]
            lo += 1
        # right side is lower: settle hi with right_max
        else:
            right_max = max(right_max, height[hi])
            water += right_max - height[hi]
            hi -= 1
    return water


assert trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]) == 6
assert trap([4, 2, 0, 3, 2, 5]) == 9
print(trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]))

6


### 2.1.6 [680. Valid Palindrome II](https://leetcode.com/problems/valid-palindrome-ii/) — ★★ · Easy

**Problem.** Can `s` become a palindrome by deleting at most one character?  
**Example.** `"abca"` → `True` (delete `c` and you get `"aba"`)  
**Spot it.** Palindrome with one mismatch allowed → on the first mismatch, try both deletions.

**Ask first**
- At most one deletion, or exactly one? → At most one; an existing palindrome is `True`.

> 💡 **Idea.** Compare from both ends; at the first mismatch you get one deletion — try skipping the left char *or* the right char and check the rest.

**Brute force:** delete each character and test, O(n²).

**Steps**
1. Walk inward while characters match.
2. At the first mismatch, test whether `s[lo+1..hi]` is a palindrome.
3. Or whether `s[lo..hi−1]` is.
4. No mismatch at all → `True`.

**Cost:** Time O(n) (one walk plus at most two more) · Space O(1)

In [33]:
def valid_palindrome_ii(s):
    """True if deleting at most one char yields a palindrome."""

    def is_pal(lo, hi):
        # plain palindrome check on s[lo..hi]
        while lo < hi:
            if s[lo] != s[hi]:
                return False
            lo, hi = lo + 1, hi - 1
        return True

    lo, hi = 0, len(s) - 1
    while lo < hi:
        # first mismatch: spend the single deletion on either side
        if s[lo] != s[hi]:
            return is_pal(lo + 1, hi) or is_pal(lo, hi - 1)
        lo, hi = lo + 1, hi - 1
    return True


assert valid_palindrome_ii("aba")
assert valid_palindrome_ii("abca")
assert not valid_palindrome_ii("abc")
print(valid_palindrome_ii("abca"))

True


### 2.1.7 [18. 4Sum](https://leetcode.com/problems/4sum/) — ★★ · Medium

**Problem.** All unique quadruplets summing to `target`.  
**Example.** `nums = [1, 0, -1, 0, -2, 2], target = 0` → `[[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]`  
**Spot it.** k-Sum → recursion that peels one element per level down to Two Sum II.

**Ask first**
- Unique quadruplets by value? → Yes → dedupe at every level.
- Can the sum overflow? → It can pass 32 bits — fine in Python, use `long` elsewhere.

> 💡 **Idea.** 3Sum with one more loop: fix two numbers, two-pointer the rest, skip duplicates at every level.

**Brute force:** four loops + a set, O(n⁴).

**Steps**
1. Sort.
2. `k_sum(start, k, goal)`: if `k == 2`, run two pointers with duplicate skipping.
3. Otherwise loop over a first element (skip duplicates) and recurse with `k − 1` and `goal − nums[i]`.
4. Prefix each returned tuple with the chosen element.

**Cost:** Time O(n^(k−1)) · Space O(k) (recursion depth, besides the output)

**Likely follow-ups**
- **Write general k-Sum.** `k_sum(start, k, target)`: if k == 2 run two pointers; else loop `i`, skip duplicates, recurse with `k − 1`. O(n^(k−1)).
- **Can you prune?** If the smallest `k` remaining values already exceed target, stop; if the largest `k` are below it, skip this `i`.
- **Can 4Sum be O(n²)?** For a yes/no or count answer, yes — pair sums in a hash map (like 4Sum II). For *unique quadruplets* deduping makes it messy; O(n³) is the expected answer.

In [ ]:
def four_sum(nums, target):
    """Unique quadruplets summing to target."""
    nums = sorted(nums)

    def k_sum(start, k, goal):
        out = []
        # base case: two-pointer 2-Sum
        if k == 2:
            lo, hi = start, len(nums) - 1
            while lo < hi:
                total = nums[lo] + nums[hi]
                if total < goal:
                    lo += 1
                elif total > goal:
                    hi -= 1
                else:
                    out.append([nums[lo], nums[hi]])
                    lo += 1
                    hi -= 1
                    # skip repeated left values
                    while lo < hi and nums[lo] == nums[lo - 1]:
                        lo += 1
            return out
        
        # peel one element and solve (k-1)-Sum on the rest
        for i in range(start, len(nums) - k + 1):
            if i > start and nums[i] == nums[i - 1]:
                continue
            for rest in k_sum(i + 1, k - 1, goal - nums[i]):
                out.append([nums[i]] + rest)
        return out

    return k_sum(0, 4, target)


assert four_sum([1, 0, -1, 0, -2, 2], 0) == [
    [-2, -1, 1, 2],
    [-2, 0, 0, 2],
    [-1, 0, 0, 1],
]
assert four_sum([2, 2, 2, 2, 2], 8) == [[2, 2, 2, 2]]
print(four_sum([1, 0, -1, 0, -2, 2], 0))

[[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]


### 2.1.8 [881. Boats to Save People](https://leetcode.com/problems/boats-to-save-people/) — ★★ · Medium

**Problem.** Each boat holds at most 2 people and weight `limit`; minimum number of boats.  
**Example.** `people = [3, 2, 2, 1], limit = 3` → `3` boats: `[1, 2]`, `[2]`, `[3]`  
**Spot it.** Pairing heaviest with lightest → sort + opposite pointers (greedy).

**Ask first**
- At most two people per boat? → Yes. With more seats this greedy stops working.
- Is everyone ≤ `limit`? → Yes, so everyone fits alone.

> 💡 **Idea.** Sort. The heaviest person always takes a boat; if the lightest fits with them, they share. Heaviest + lightest is the best partner anyone could give the heaviest.

**Brute force:** try all pairings, exponential.

**Steps**
1. Sort weights.
2. The heaviest (`hi`) always boards a new boat.
3. If the lightest (`lo`) fits too, board them as well.
4. Count boats until everyone is on.

**Cost:** Time O(n log n) (sorting dominates; the pointer pass is O(n)) · Space O(1) extra

**Likely follow-ups**
- **Why is pairing heaviest with lightest optimal?** If the lightest can't ride with the heaviest, nobody can — heaviest goes alone. If they can, swapping any other partner for the lightest never hurts.
- **Boats hold up to k people?** Becomes bin packing (NP-hard). Use first-fit decreasing as a heuristic, or bitmask DP for small n.
- **Weights in a small range (≤ 3·10⁴)?** Counting sort the weights → O(n + limit) instead of O(n log n).

In [35]:
def num_rescue_boats(people, limit):
    """Fewest two-seat boats needed."""
    people = sorted(people)
    lo, hi, boats = 0, len(people) - 1, 0
    # <= because a single person may still need a boat
    while lo <= hi:
        # the lightest joins the heaviest if they fit together
        if people[lo] + people[hi] <= limit:
            lo += 1
        # the heaviest always leaves on this boat
        hi -= 1
        boats += 1
    return boats


assert num_rescue_boats([1, 2], 3) == 1
assert num_rescue_boats([3, 2, 2, 1], 3) == 3
assert num_rescue_boats([3, 5, 3, 4], 5) == 4
print(num_rescue_boats([3, 2, 2, 1], 3))

3


## 2.2 Same-direction pointers

Both pointers go left to right. One reads everything; the other moves only when something is kept or matched. Ask: *when does the slow pointer move?*

### 2.2.1 [392. Is Subsequence](https://leetcode.com/problems/is-subsequence/) — ★★ · Easy

**Problem.** Is `s` a subsequence of `t`?  
**Example.** `s = "abc", t = "ahbgdc"` → `True`; `s = "axc"` → `False`  
**Spot it.** Match in order, skipping allowed → one pointer per string.

**Ask first**
- Many `s` queries on the same `t`? → Precompute each letter's positions in `t` and binary-search the next one.

> 💡 **Idea.** Walk `t`; every time its character matches the next needed character of `s`, move forward in `s`.

**Brute force:** try every way of picking letters, exponential.

**Steps**
1. Pointer `i` into `s` starts at 0.
2. Walk through `t`; when `t`'s character equals `s[i]`, advance `i`.
3. `s` is a subsequence iff `i` reaches `len(s)`.

**Cost:** Time O(|t|) (one pass over `t`) · Space O(1)

In [36]:
def is_subsequence(s, t):
    """True if s can be obtained by deleting chars from t."""
    i = 0
    for ch in t:
        # greedy: consume s[i] at its earliest match in t
        if i < len(s) and s[i] == ch:
            i += 1
    return i == len(s)


assert is_subsequence("abc", "ahbgdc")
assert not is_subsequence("axc", "ahbgdc")
assert is_subsequence("", "abc")
print(is_subsequence("abc", "ahbgdc"))

True


### 2.2.2 [151. Reverse Words in a String](https://leetcode.com/problems/reverse-words-in-a-string/) — ★★ · Medium

**Problem.** Reverse the order of words; collapse extra spaces.  
**Example.** `"the sky is blue"` → `"blue is sky the"`  
**Spot it.** In Python `split()` handles spaces; in-place version = reverse whole, then each word.

**Ask first**
- Leading, trailing or repeated spaces? → Yes; the output has single spaces only.
- Can I use `split()`? → Usually yes in Python. If they want O(1) space, you need the in-place version below.

> 💡 **Idea.** Split on whitespace (drops extra spaces), reverse the list, join with single spaces.

**Brute force:** build words manually character by character — same O(n), just more code.

**Steps**
1. Split on whitespace (drops extra spaces).
2. Reverse the list of words by swapping the first and last, then moving inwards.
3. Join with single spaces.

**Cost:** Time O(n) (split, reverse and join are each linear) · Space O(n) (the word list)

**Likely follow-ups**
- **In place with O(1) extra space (mutable char array)?** Squash extra spaces with a write pointer, reverse the whole array, then reverse each word. O(n), O(1).
- **Input has no extra spaces and is a char array (LC 186)?** Just reverse all, then reverse each word.
- **Reverse the letters inside each word but keep word order (LC 557)?** Reverse each word in place; skip the full reverse.

In [37]:
def reverse_list(items):
    """Reverse a list in place by swapping from both ends."""
    lo, hi = 0, len(items) - 1
    while lo < hi:
        items[lo], items[hi] = items[hi], items[lo]
        lo += 1
        hi -= 1


def reverse_words(s):
    """Words of s in reverse order, single-spaced."""
    # split() without arguments also collapses repeated spaces
    words = s.split()
    reverse_list(words)
    return " ".join(words)

assert reverse_words("the sky is blue") == "blue is sky the"
assert reverse_words("  hello world  ") == "world hello"
assert reverse_words("a good   example") == "example good a"
print(reverse_words("the sky is blue"))

blue is sky the


---
# 3) Sliding Window

**The gist.** Look at a contiguous block through a window that only slides right. The right edge grows it; the left edge shrinks it when a rule breaks. Each element enters once and leaves once → O(n).

> **Think of it like…** a caterpillar on a branch: the head stretches forward to eat; when the body gets too long, the tail pulls in. It never crawls backwards.

```python
left = 0
for right, x in enumerate(arr):
    add(x)                              # head moves forward
    while window_is_invalid():          # tail catches up
        remove(arr[left]); left += 1
    best = max(best, right - left + 1)  # longest valid window
```

For the *shortest* valid window, update the answer *inside* the while loop instead.

**Spot it:** "longest / shortest substring or subarray such that…", "at most k distinct / flips / replacements", "every window of size k", all values non-negative.

**The test before you use it:** once a window is invalid, is every bigger window containing it invalid too? If not (e.g. negative numbers), use prefix sums + a hash map instead.

**Classic bugs:** moving `left` backwards (`left = last[ch] + 1` without `max`) · leaving zero-count keys in the map when you count distinct keys.

## 3.1 Variable-size windows

Right edge always moves; left edge moves only to fix the window. Before coding, name the **rule** for a valid window and the **state** that checks it in O(1) (a sum, a count map, a zero counter).

### 3.1.1 [121. Best Time to Buy and Sell Stock](https://leetcode.com/problems/best-time-to-buy-and-sell-stock/) — ★★★ · Easy

**Problem.** One buy then one later sell; maximum profit (0 if none).  
**Example.** `[7, 1, 5, 3, 6, 4]` → `5` (buy at 1, sell at 6)  
**Spot it.** Best pair `i < j` maximising `p[j] − p[i]` → track the running minimum.

**Ask first**
- Exactly one buy and one sell? → At most one; return 0 if prices only fall.

> 💡 **Idea.** Track the cheapest price so far; selling today earns `price − cheapest`.

**Brute force:** try every buy/sell pair, O(n²).

**Steps**
1. Keep `min_price` = cheapest price seen so far.
2. For each price, update `min_price`.
3. Profit if sold today = `price − min_price`; keep the best.

**Cost:** Time O(n) (one pass) · Space O(1) (two variables)

In [38]:
def max_profit(prices):
    """Best single-trade profit."""
    min_price, best = float("inf"), 0
    for p in prices:
        # cheapest buy available up to today
        min_price = min(min_price, p)
        # profit if we sell today
        best = max(best, p - min_price)
    return best


assert max_profit([7, 1, 5, 3, 6, 4]) == 5
assert max_profit([7, 6, 4, 3, 1]) == 0
print(max_profit([7, 1, 5, 3, 6, 4]))

5


### 3.1.2 [3. Longest Substring Without Repeating Characters](https://leetcode.com/problems/longest-substring-without-repeating-characters/) — ★★★ · Medium

**Problem.** Length of the longest substring with all distinct characters.  
**Example.** `"abcabcbb"` → `3` (`"abc"`)  
**Spot it.** Longest contiguous block with a property → sliding window.

**Ask first**
- What characters? → Any (letters, digits, spaces, symbols), so use a dict, not 26 slots.

> 💡 **Idea.** Grow the window to the right. When the new char is already inside, jump `left` to just past its last position.

**Brute force:** check every substring for repeats, O(n³) (O(n²) with a set).

**Steps**
1. Map each character to its last index.
2. For each `right`, if `s[right]` was last seen at or after `left`, move `left` to that index + 1.
3. Update the last index of `s[right]`.
4. Record `right − left + 1`.

**Cost:** Time O(n) (`right` moves n times and `left` only moves forward) · Space O(alphabet) (the last-index map)

**Likely follow-ups**
- **At most k distinct characters (LC 340)?** Count map in the window; while `len(count) > k`, shrink from the left. O(n), O(k).
- **Return the substring itself?** Store `best_left` with the best length and slice at the end.
- **Unicode / emoji input?** Hash map keyed by code point works as-is; if "character" means a visible glyph, split into grapheme clusters first.
- **Why `max(left, last[ch] + 1)`?** The old position might be left of the window already — without `max`, `left` jumps backwards.

In [39]:
def length_of_longest_substring(s):
    """Longest substring without repeated characters."""
    # char -> index of its latest occurrence
    last = {}
    left = best = 0
    for right, ch in enumerate(s):
        # repeat inside the window: jump left past the old copy (never backwards)
        if ch in last:
            left = max(left, last[ch] + 1)
        last[ch] = right
        best = max(best, right - left + 1)
    return best


assert length_of_longest_substring("abcabcbb") == 3
assert length_of_longest_substring("bbbbb") == 1
assert length_of_longest_substring("pwwkew") == 3
assert length_of_longest_substring("abba") == 2
print(length_of_longest_substring("abcabcbb"))

3


### 3.1.3 [424. Longest Repeating Character Replacement](https://leetcode.com/problems/longest-repeating-character-replacement/) — ★★★ · Medium

**Problem.** Replace at most `k` characters; longest substring of one repeated letter.  
**Example.** `s = "AABABBA", k = 1` → `4` (change one letter to get `"AAAA"` or `"BBBB"`)  
**Spot it.** Window is valid while `size − (count of most frequent char) ≤ k`.

**Ask first**
- Only uppercase letters? → Yes, 26 counts. A bigger alphabet just means a dict.

> 💡 **Idea.** A window is fine if `size − (count of its most common letter) ≤ k` — that many replacements turn it all into that letter. Grow; shrink by one when it breaks.

**Brute force:** for every substring, check `len − max_count ≤ k`, O(n²·26).

**Steps**
1. Grow the window and count the new letter.
2. Track `max_freq`, the highest single-letter count seen.
3. If `size − max_freq > k`, slide `left` forward by one (the window keeps its size).
4. The final window size is the answer.

**Cost:** Time O(n) (each step does O(1) work) · Space O(26) (letter counts)

**Likely follow-ups**
- **Why is it OK never to lower `max_freq` when shrinking?** The answer only grows when a window *bigger* than the best is valid, which needs a *real* bigger `max_freq`. A stale value can't create a false answer.
- **Must replace with one specific letter `c`?** Same as Max Consecutive Ones III: count non-`c` letters in the window, keep it ≤ k.
- **Return the actual substring?** Remember `left` when you update the best length.

In [ ]:
def character_replacement(s, k):
    """Longest same-letter substring after at most k replacements."""
    count = Counter()
    left = max_freq = 0
    for right, ch in enumerate(s):
        count[ch] += 1

        # best single-letter count ever seen in a window
        max_freq = max(max_freq, count[ch])

        # letters to replace exceed k: slide (window size stays the same)
        if right - left + 1 - max_freq > k:
            count[s[left]] -= 1
            left += 1

    # the window never shrank, so its final size is the best size
    return len(s) - left


assert character_replacement("ABAB", 2) == 4
assert character_replacement("AABABBA", 1) == 4
print(character_replacement("AABABBA", 1))

4


### 3.1.4 [76. Minimum Window Substring](https://leetcode.com/problems/minimum-window-substring/) — ★★★ · Hard

**Problem.** Smallest substring of `s` that contains every character of `t` (with multiplicity).  
**Example.** `s = "ADOBECODEBANC", t = "ABC"` → `"BANC"`  
**Spot it.** Shortest window covering a requirement → expand until valid, shrink while valid.

**Ask first**
- Do duplicates in `t` count? → Yes, `"aa"` needs two `a`s — so track counts and a `missing` total.

> 💡 **Idea.** Expand right until the window has everything `t` needs; then shrink from the left while it still does, recording the smallest.

**Brute force:** check every substring, O(n²·|Σ|).

**Steps**
1. `need` = counts of `t`; `missing` = `len(t)`.
2. Expand `right`: if `need[ch] > 0`, one requirement is met (`missing −= 1`); decrement `need[ch]`.
3. While `missing == 0`, record the window, then return `s[left]` to `need` and move `left`.
4. If returning it makes `need > 0`, the window is short again (`missing += 1`).

**Cost:** Time O(|s| + |t|) (each index enters and leaves once) · Space O(alphabet) (the `need` map)

**Likely follow-ups**
- **Many `t` queries on the same `s`?** Precompute positions per character; still O(n) per query in general.
- **`t` must appear as a *subsequence* in order (LC 727)?** Different tool: for each start, greedily match forward (or DP). O(n·m).

In [41]:
def min_window(s, t):
    """Shortest substring of s containing all of t."""
    # need[c] > 0: still required; < 0: surplus inside the window
    need = Counter(t)
    missing = len(t)
    left = 0
    best = (0, float("inf"))
    for right, ch in enumerate(s):
        # a still-required char reduces the shortfall
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        # window covers t: shrink it as far as possible
        while missing == 0:
            if right - left < best[1] - best[0]:
                best = (left, right)
            need[s[left]] += 1
            # dropped a required char: window no longer covers t
            if need[s[left]] > 0:
                missing += 1
            left += 1
    return "" if best[1] == float("inf") else s[best[0] : best[1] + 1]


assert min_window("ADOBECODEBANC", "ABC") == "BANC"
assert min_window("a", "a") == "a"
assert min_window("a", "aa") == ""
print(min_window("ADOBECODEBANC", "ABC"))

BANC


### 3.1.5 [209. Minimum Size Subarray Sum](https://leetcode.com/problems/minimum-size-subarray-sum/) — ★★ · Medium

**Problem.** Positive integers; length of the shortest subarray with sum ≥ `target` (0 if none).  
**Example.** `target = 7, nums = [2, 3, 1, 2, 4, 3]` → `2` (the piece `[4, 3]`)  
**Spot it.** Positive values → window sum is monotone → shortest-window template.

**Ask first**
- Are all values positive? → Yes. With negatives the window breaks (see follow-up).
- `≥ target` or exactly `target`? → `≥`. Exactly would need prefix sums + a map.

> 💡 **Idea.** All numbers are positive, so growing the window only raises the sum. Expand right; while the sum ≥ target, record the length and shrink from the left.

**Brute force:** check every subarray, O(n²).

**Steps**
1. Add `nums[right]` to the running sum.
2. While the sum ≥ target, record the length and remove `nums[left]`.
3. Return the best length, or 0 if none.

**Cost:** Time O(n) (each element is added and removed once) · Space O(1)

**Likely follow-ups**
- **Give an O(n log n) solution.** Prefix sums are increasing; for each `i` binary-search the first `j` with `prefix[j] ≥ prefix[i] + target`.
- **Values can be negative (LC 862)?** Prefix sums + a monotonic deque of increasing prefix values; pop from the front while `P[j] − P[front] ≥ target`. O(n).
- **Return the subarray itself?** Track the `left, right` that gave the best length.

In [42]:
def min_sub_array_len(target, nums):
    """Shortest subarray length with sum >= target."""
    left = total = 0
    best = len(nums) + 1
    for right, x in enumerate(nums):
        total += x
        # valid: record, then try a shorter window
        while total >= target:
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return best if best <= len(nums) else 0


assert min_sub_array_len(7, [2, 3, 1, 2, 4, 3]) == 2
assert min_sub_array_len(4, [1, 4, 4]) == 1
assert min_sub_array_len(11, [1, 1, 1, 1, 1, 1, 1, 1]) == 0
print(min_sub_array_len(7, [2, 3, 1, 2, 4, 3]))

2


### 3.1.6 [1004. Max Consecutive Ones III](https://leetcode.com/problems/max-consecutive-ones-iii/) — ★ · Medium

**Problem.** Binary array; flip at most `k` zeros; longest run of 1s.  
**Example.** `nums = [1,1,1,0,0,0,1,1,1,1,0], k = 2` → `6`  
**Spot it.** "At most k bad elements in the window" → longest-window template counting zeros.

**Ask first**
- Can `k` be 0 or more than the number of zeros? → Yes, both — the window handles them without special cases.

> 💡 **Idea.** Rephrase: longest window with **at most k zeros**. Expand; when zeros exceed k, move `left` past a zero.

**Brute force:** for each start, extend while zeros ≤ k, O(n²).

**Steps**
1. Count zeros in the window as `right` advances.
2. While zeros > k, shrink from the left (decrement if a zero leaves).
3. Record the window size.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Input is an infinite stream (can't look back at `nums[left]`)?** Keep a queue of the last `k + 1` zero positions. When a new zero arrives and the queue is full, the window starts after the oldest zero. O(k) memory.
- **Only one flip allowed (LC 487)?** Same code with k = 1, or track `prev_run` and `cur_run` lengths in O(1) memory.
- **Return the indices to flip?** Track the best window, then return the zero positions inside it.

In [43]:
def longest_ones(nums, k):
    """Longest 1-run after flipping at most k zeros."""
    left = zeros = best = 0
    for right, x in enumerate(nums):
        zeros += x == 0
        # too many zeros to flip: shrink
        while zeros > k:
            zeros -= nums[left] == 0
            left += 1
        best = max(best, right - left + 1)
    return best


assert longest_ones([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2) == 6
assert longest_ones([0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1], 3) == 10
print(longest_ones([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2))

6


### 3.1.7 [1493. Longest Subarray of 1's After Deleting One Element](https://leetcode.com/problems/longest-subarray-of-1s-after-deleting-one-element/) — ★ · Medium

**Problem.** Delete exactly one element; longest subarray of only 1s.  
**Example.** `[1, 1, 0, 1]` → `3` (delete the 0)  
**Spot it.** Same as Max Consecutive Ones III with `k = 1`, minus the deleted element.

**Ask first**
- Must I delete even if all are 1s? → Yes, so `[1,1,1]` → 2. The `−1` handles it.

> 💡 **Idea.** Longest window with at most one zero, minus 1 (you must delete one element).

**Brute force:** delete each element and scan, O(n²).

**Steps**
1. Longest window with at most one zero.
2. Record `right − left` (window size minus the deleted element).
3. All-ones input still deletes one 1, which the formula already handles.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Delete up to k elements?** Window with at most k zeros; deleted zeros don't count, so the answer is window length − zeros inside it.
- **Stream version?** Track `prev_run` (ones before the last zero) and `cur_run`; answer = `max(prev + cur)`. O(1) memory.

In [44]:
def longest_subarray_after_delete(nums):
    """Longest all-ones subarray after deleting one element."""
    left = zeros = best = 0
    for right, x in enumerate(nums):
        zeros += x == 0
        # at most one zero (the one we delete)
        while zeros > 1:
            zeros -= nums[left] == 0
            left += 1
        # size minus the mandatory deletion
        best = max(best, right - left)
    return best


assert longest_subarray_after_delete([1, 1, 0, 1]) == 3
assert longest_subarray_after_delete([0, 1, 1, 1, 0, 1, 1, 0, 1]) == 5
assert longest_subarray_after_delete([1, 1, 1]) == 2
print(longest_subarray_after_delete([0, 1, 1, 1, 0, 1, 1, 0, 1]))

5


### 3.1.8 [904. Fruit Into Baskets](https://leetcode.com/problems/fruit-into-baskets/) — ★ · Medium

**Problem.** Longest subarray containing at most 2 distinct values.  
**Example.** `[1, 2, 1]` → `3`; `[0, 1, 2, 2]` → `3`  
**Spot it.** "At most k distinct" → window with a count map; shrink while map has > k keys.

**Ask first**
- Must I pick contiguous trees? → Yes, left to right with no skipping — that's what makes it a window.

> 💡 **Idea.** Rephrase: longest subarray with **at most 2 distinct values**. Count map in the window; while it has 3 keys, shrink.

**Brute force:** for each start, extend until a third type, O(n²).

**Steps**
1. Add `fruits[right]` to the count map.
2. While the map has more than two keys, remove `fruits[left]`, deleting keys that hit zero.
3. Record the window size.

**Cost:** Time O(n) · Space O(1) (at most three keys at any time)

**Likely follow-ups**
- **k baskets?** Same code with `len(count) > k` — this is LC 340. O(n) time, O(k) space.
- **Return which fruit types you picked?** Keep the best window's `left, right`; the types are the keys of the count at that moment.
- **O(1) space without a map?** For 2 types: track the last two types and the length of the last run of the latest type.

In [45]:
def total_fruit(fruits):
    """Longest subarray with at most two distinct values."""
    count = Counter()
    left = best = 0
    for right, f in enumerate(fruits):
        count[f] += 1
        # three kinds: drop from the left until one kind disappears
        while len(count) > 2:
            count[fruits[left]] -= 1
            if count[fruits[left]] == 0:
                del count[fruits[left]]
            left += 1
        best = max(best, right - left + 1)
    return best


assert total_fruit([1, 2, 1]) == 3
assert total_fruit([0, 1, 2, 2]) == 3
assert total_fruit([1, 2, 3, 2, 2]) == 4
print(total_fruit([1, 2, 3, 2, 2]))

4


## 3.2 Fixed-size windows

Window size is fixed at `k`: add `arr[right]`, drop `arr[right - k]`, no while loop. The only question is *what summary of the window you keep* (sum, count map, deque).

### 3.2.1 [567. Permutation in String](https://leetcode.com/problems/permutation-in-string/) — ★★ · Medium

**Problem.** Does `s2` contain a permutation of `s1` as a substring?  
**Example.** `s1 = "ab", s2 = "eidbaooo"` → `True` (`"ba"` is inside)  
**Spot it.** Anagram of a fixed-length pattern → fixed window of size `len(s1)` with counts.

**Ask first**
- Only lowercase letters? → Yes, so 26-count arrays. Unicode → dict counts plus a `matches` counter.

> 💡 **Idea.** Slide a window of length `len(s1)` over `s2` and compare letter counts. Track how many of the 26 letters match so each step is O(1).

**Brute force:** generate every permutation of `s1` and search, O(n!·m).

**Steps**
1. Count `s1` and the first `k` characters of `s2`.
2. If equal, return `True`.
3. Slide: add `s2[right]`, remove `s2[right − k]`, compare again.

**Cost:** Time O(n) (each comparison is over at most 26 keys) · Space O(26)

**Likely follow-ups**
- **Unicode characters?** Use dicts; keep a count of keys whose counts differ and update it as one char enters and one leaves. O(n) time, O(distinct chars) space.
- **Return all start positions (LC 438)?** Same window — collect each index where counts match.
- **Allow up to k mismatched letters?** Track the total absolute difference between counts; window is OK if it's ≤ 2k.

In [ ]:
def check_inclusion(s1, s2):
    """True if some substring of s2 is an anagram of s1."""
    k = len(s1)
    target = Counter(s1)
    window = Counter()

    for right in range(len(s2)):
        # new char enters on the right
        window[s2[right]] += 1    
        # frame is now too wide          
        if right >= k:       
            # old char leaves on the left               
            window[s2[right - k]] -= 1      
            
        if window == target:
            return True
    return False


assert check_inclusion("ab", "eidbaooo")
assert not check_inclusion("ab", "eidboaoo")
print(check_inclusion("ab", "eidbaooo"))

True


### 3.2.2 [438. Find All Anagrams in a String](https://leetcode.com/problems/find-all-anagrams-in-a-string/) — ★★ · Medium

**Problem.** All start indices in `s` of anagrams of `p`.  
**Example.** `s = "cbaebabacd", p = "abc"` → `[0, 6]` (`"cba"` and `"bac"`)  
**Spot it.** Permutation in String, collecting every match instead of stopping.

**Ask first**
- Can matches overlap? → Yes, report every start index.

> 💡 **Idea.** Same as Permutation in String, but record every start index where the window's counts equal `p`'s.

**Brute force:** sort/count every window from scratch, O(n·m).

**Steps**
1. Add `s[right]` to the window counts.
2. Once `right ≥ k`, remove `s[right − k]`.
3. If the counts equal `p`'s, record `right − k + 1`.

**Cost:** Time O(n) · Space O(26)

**Likely follow-ups**
- **Return the substrings instead of indices?** `[s[i:i+m] for i in starts]`.
- **Many patterns on the same text?** Group patterns by length; slide one window per length and look up its count-tuple in a set of pattern keys.
- **Unicode?** Dict counts + a `matches` counter, same O(n).

In [47]:
def find_anagrams(s, p):
    """Start indices of p's anagrams in s."""
    k = len(p)
    target, window = Counter(p), Counter()
    out = []
    for right, ch in enumerate(s):
        window[ch] += 1
        # keep the window at length k
        if right >= k:
            window[s[right - k]] -= 1
        if window == target:
            out.append(right - k + 1)
    return out


assert find_anagrams("cbaebabacd", "abc") == [0, 6]
assert find_anagrams("abab", "ab") == [0, 1, 2]
print(find_anagrams("cbaebabacd", "abc"))

[0, 6]


### 3.2.3 [1423. Maximum Points You Can Obtain from Cards](https://leetcode.com/problems/maximum-points-you-can-obtain-from-cards/) — ★ · Medium

**Problem.** Take exactly `k` cards from the two ends; maximise the sum.  
**Example.** `cardPoints = [1, 2, 3, 4, 5, 6, 1], k = 3` → `12` (take the last three cards: 5 + 6 + 1)  
**Spot it.** The `k` cards always sit as a prefix plus a suffix, and moving one card from one side to the other changes the sum by a single add and a single subtract.

**Ask first**
- Must you take exactly k? → Yes; k = n means take everything.

> 💡 **Idea.** You take `i` cards from the left and `k − i` from the right. Start with all k from the right, then move one card at a time to the left side, updating the sum. (Same thing: the cards you *leave* are one middle block of `n − k`.)

**Brute force:** try all k + 1 ways to split left/right, recomputing sums, O(k²) (O(k) with prefix sums).

**Steps**
1. Let `size = n − k`. The last `k` cards start at index `size`.
2. `total` = sum of those last `k` cards. This is the split with 0 cards from the left.
3. For `i` in `0..k−1`: add `card_points[i]` (take from the left), subtract `card_points[size + i]` (give back from the right), and keep the best.

**Cost:** Time O(k) · Space O(1)

**Likely follow-ups**
- **Middle-window version?** The cards left behind form one block of `n − k`; find the cheapest such block with a fixed window, answer = total − that. O(n).
- **Cards can only be taken from one end per turn, with a weight per turn (Google variant)?** Becomes DP over `(left_taken, right_taken)` — O(k²) (like LC 1770).

In [ ]:
def max_score(card_points, k):
    """Best sum of k cards taken from the ends."""
    n = len(card_points)
    size = n - k
    # start by taking all k cards from the right
    total = best = sum(card_points[size:])
    
    for i in range(k):
        # take left card i, give back right card size + i
        total += card_points[i] - card_points[size + i]
        best = max(best, total)
    return best


assert max_score([1, 2, 3, 4, 5, 6, 1], 3) == 12
assert max_score([2, 2, 2], 2) == 4
assert max_score([9, 7, 7, 9, 7, 7, 9], 7) == 55
print(max_score([1, 2, 3, 4, 5, 6, 1], 3))

12


### 3.2.4 [239. Sliding Window Maximum](https://leetcode.com/problems/sliding-window-maximum/) — ★★ · Hard

**Problem.** Maximum of every window of size `k`.  
**Example.** `nums = [1, 3, -1, -3, 5, 3, 6, 7], k = 3` → `[3, 3, 5, 5, 6, 7]`  
**Spot it.** Window max with O(1) amortised updates → monotonic decreasing deque of indices.

**Ask first**
- Duplicates? → Yes; pop only *strictly* smaller from the back (or ≤ — both work, just be consistent).

> 💡 **Idea.** Keep a deque of indices whose values are decreasing. Front = current max. Pop from the back anything smaller than the new value (it can never be the max again); pop the front when it leaves the window.

**Brute force:** max of each window, O(n·k).

**Steps**
1. Keep a deque of indices whose values decrease front to back.
2. Before pushing `i`, pop back indices with values ≤ `nums[i]`.
3. Pop the front if it has slid out of the window.
4. Once the first window is full, the front is the max.

**Cost:** Time O(n) (each index is pushed and popped at most once) · Space O(k) (the deque)

**Likely follow-ups**
- **Sliding window *median* (LC 480)?** Two heaps with lazy deletion (or a sorted list). O(n log k).
- **Stream with an unknown k per query?** Sparse table for O(1) range-max after O(n log n) prep (static), or a segment tree with updates.

In [49]:
def max_sliding_window(nums, k):
    """Maximum of each length-k window."""
    # indices with decreasing values; front = current max
    dq = deque()
    out = []
    for i, x in enumerate(nums):
        # smaller values behind x can never be a max again
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        # front index left the window
        if dq[0] == i - k:
            dq.popleft()
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out


assert max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert max_sliding_window([1], 1) == [1]
print(max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3))

[3, 3, 5, 5, 6, 7]


---
# 4) Prefix Sums

**The gist.** Store running totals once; the sum of any range is the difference of two totals. Add a hash map and "does some subarray sum to k?" becomes "have I seen the total `current − k` before?".

> **Think of it like…** a car's odometer. Distance between two towns = difference of the two readings. No re-measuring the road.

`prefix[i] = nums[0] + … + nums[i-1]`, so `sum(nums[i:j]) = prefix[j] − prefix[i]`.

**Spot it:** "subarray sum **equals** k", "number of subarrays with…", "divisible by k", negatives present (window fails), many range queries, many "add v to [l, r]" updates (difference array).

**Classic bugs:** forgetting the seed `{0: 1}` (count) or `{0: -1}` (first index) · storing the *latest* index when you want the *longest* subarray · updating the map before the lookup.

## 4.1 Prefix sum with hash map

Core equation: subarray `(i, j]` sums to `k` ⇔ `prefix[i] = prefix[j] − k`. So at each `j`, ask the map about `prefix[j] − k`. Store **counts** to count subarrays, **first indices** to maximise length.

### 4.1.1 [560. Subarray Sum Equals K](https://leetcode.com/problems/subarray-sum-equals-k/) — ★★ · Medium

**Problem.** Count contiguous subarrays summing to `k` (values may be negative).  
**Example.** `nums = [1, 2, 3], k = 3` → `2` (`[1, 2]` and `[3]`)  
**Spot it.** Negative values break sliding windows → prefix sum + hash map of prefix counts.

**Ask first**
- Can values be negative? → Yes → sliding window fails; prefix sums + map are needed. If all are non-negative, a window works with O(1) space.

> 💡 **Idea.** At each position, the number of subarrays ending here with sum `k` = how many earlier prefix sums equal `current − k`. Keep a count map of prefix sums.

**Brute force:** sum every subarray, O(n²).

**Steps**
1. Seed the map with `{0: 1}` (the empty prefix).
2. Add each element to the running prefix.
3. Add `count[prefix − k]` to the answer.
4. Then record the current prefix in the map.

**Cost:** Time O(n) (one pass of O(1) map operations) · Space O(n) (up to n distinct prefixes)

**Likely follow-ups**
- **All values non-negative — O(1) space?** Sliding window (careful with zeros: count windows with sum ≤ k minus sum ≤ k−1, i.e. "at most" trick).
- **Return the subarrays, not the count?** Store a list of indices per prefix sum; for each `j`, every stored `i` gives `(i+1, j)`. Output can be O(n²).
- **Longest subarray with sum k (LC 325)?** Store the *first* index of each prefix sum; length = `j − first[cur − k]`.
- **2D: submatrices summing to k (LC 1074)?** Fix top and bottom rows, collapse columns into a 1D array, run this. O(rows²·cols).

In [50]:
def subarray_sum(nums, k):
    """Count of subarrays with sum k."""
    # prefix value -> how many times it occurred; 0 = empty prefix
    seen = Counter({0: 1})
    prefix = total = 0
    for x in nums:
        prefix += x
        # each earlier prefix equal to prefix - k starts a valid subarray
        total += seen[prefix - k]
        # record after the lookup so empty subarrays are not counted
        seen[prefix] += 1
    return total


assert subarray_sum([1, 1, 1], 2) == 2
assert subarray_sum([1, 2, 3], 3) == 2
assert subarray_sum([1, -1, 0], 0) == 3
print(subarray_sum([1, 1, 1], 2))

2


### 4.1.2 [525. Contiguous Array](https://leetcode.com/problems/contiguous-array/) — ★★ · Medium

**Problem.** Binary array; longest subarray with equal numbers of 0s and 1s.  
**Example.** `[0, 1, 0]` → `2` (`[0, 1]` or `[1, 0]`)  
**Spot it.** Map 0 → −1; equal counts ⇔ subarray sum 0 ⇔ two equal prefix sums.

**Ask first**
- Return length or the subarray? → Length (store the start too if they want the subarray).

> 💡 **Idea.** Turn 0 into −1. Equal 0s and 1s ⇔ subarray sum 0 ⇔ same prefix sum seen before. Store the *first* index of each sum to get the longest.

**Brute force:** count zeros and ones for every subarray, O(n²).

**Steps**
1. Running total: +1 for a 1, −1 for a 0.
2. Seed `first = {0: -1}`.
3. If the total was seen before, the gap `i − first[total]` is balanced; keep the max.
4. Otherwise store `i` as the first index of this total.

**Cost:** Time O(n) · Space O(n) (the first-index map)

**Likely follow-ups**
- **Three values (0, 1, 2) with equal counts?** Key the map by the pair of differences `(c1 − c0, c2 − c1)`; same first-index trick.
- **Return the subarray itself?** Keep `best_start` along with the best length.
- **Count such subarrays instead of the longest?** Store counts of each prefix sum instead of first index (like 560).

In [51]:
def find_max_length(nums):
    """Longest balanced 0/1 subarray."""
    # prefix value -> first index where it appeared (-1 = before the array)
    first = {0: -1}
    prefix = best = 0
    for i, x in enumerate(nums):
        # 1 counts +1, 0 counts -1
        prefix += 1 if x else -1
        # same total again: the stretch in between is balanced
        if prefix in first:
            best = max(best, i - first[prefix])
        # keep only the first index (gives the longest stretch)
        else:
            first[prefix] = i
    return best


assert find_max_length([0, 1]) == 2
assert find_max_length([0, 1, 0]) == 2
assert find_max_length([0, 1, 1, 1, 1, 1, 0, 0, 0]) == 6
print(find_max_length([0, 1, 1, 1, 1, 1, 0, 0, 0]))

6


### 4.1.3 [523. Continuous Subarray Sum](https://leetcode.com/problems/continuous-subarray-sum/) — ★ · Medium

**Problem.** Is there a subarray of length ≥ 2 whose sum is a multiple of `k`?  
**Example.** `nums = [23, 2, 4, 6, 7], k = 6` → `True` (`[2, 4]` sums to 6)  
**Spot it.** Same remainder twice, at least two indices apart.

**Ask first**
- Length at least 2? → Yes → store the *earliest* index and check `i − first ≥ 2`.
- Is 0 a multiple of k? → Yes, so seed `{0: -1}`.

> 💡 **Idea.** Two prefix sums with the same remainder mod k ⇒ the part between them is a multiple of k. Store the first index of each remainder; need a gap of ≥ 2.

**Brute force:** sum every subarray of length ≥ 2, O(n²).

**Steps**
1. Seed `first = {0: -1}`.
2. Update the running remainder.
3. Seen before and `i − first ≥ 2` → `True`.
4. Not seen → store `i`; never overwrite an earlier index.

**Cost:** Time O(n) · Space O(k)

**Likely follow-ups**
- **k can be 0?** Then "multiple of 0" means sum 0 — use the raw prefix sum as the key instead of the remainder.
- **Negative numbers?** Python's `%` already returns a non-negative remainder; in Java/C++ use `((sum % k) + k) % k`.
- **Count subarrays divisible by k (LC 974)?** Store counts per remainder and add `count[rem]` at each step (no length rule).

In [52]:
def check_subarray_sum(nums, k):
    """True if a length>=2 subarray sums to a multiple of k."""
    # remainder -> first index where it occurred
    first = {0: -1}
    prefix = 0
    for i, x in enumerate(nums):
        prefix = (prefix + x) % k
        if prefix in first:
            # same remainder at least two positions back
            if i - first[prefix] >= 2:
                return True
        else:
            first[prefix] = i
    return False


assert check_subarray_sum([23, 2, 4, 6, 7], 6)
assert check_subarray_sum([23, 2, 6, 4, 7], 6)
assert not check_subarray_sum([23, 2, 6, 4, 7], 13)
print(check_subarray_sum([23, 2, 4, 6, 7], 6))

True


## 4.2 Range queries and difference arrays

No hash map here: precompute a prefix array once and answer each query in O(1). 2D adds inclusion–exclusion. A **difference array** runs the idea backwards for range *updates*: mark `+v` at the start, `−v` after the end, prefix-sum at the end.

### 4.2.1 [304. Range Sum Query 2D - Immutable](https://leetcode.com/problems/range-sum-query-2d-immutable/) — ★★ · Medium

**Problem.** Many queries for the sum of a sub-rectangle.  
**Example.** `matrix = [[1, 2], [3, 4]]`: `sum_region(0, 0, 1, 1)` → `10`  
**Spot it.** 2D prefix sums with inclusion–exclusion.

**Ask first**
- Will the matrix change between queries? → No. With updates you need a 2D Fenwick tree (LC 308).

> 💡 **Idea.** Precompute `P[r][c]` = sum of the rectangle from (0, 0) to (r−1, c−1). Any rectangle = big − top − left + top-left overlap (added back).

**Brute force:** sum every cell of each query, O(rows·cols) per query.

**Steps**
1. `P[r+1][c+1]` = sum of `(0,0)..(r,c)`, built from its up, left and up-left neighbours.
2. Query = `P[r2+1][c2+1] − P[r1][c2+1] − P[r2+1][c1] + P[r1][c1]`.

**Cost:** Time O(mn) build, O(1) per query · Space O(mn) (the table)

**Likely follow-ups**
- **Cells get updated (LC 308)?** 2D Fenwick tree: update and query both O(log m · log n). Simpler: one 1D Fenwick per row → O(m log n) query.
- **1D version (LC 303)?** `prefix[j+1] − prefix[i]`.
- **Huge sparse matrix?** Store only non-zero points, sort by row, and use an offline sweep with a 1D Fenwick over columns.

In [53]:
class NumMatrix:
    """Static 2D range-sum structure."""

    def __init__(self, matrix):
        rows, cols = len(matrix), len(matrix[0])
        # padded table: row 0 and column 0 stay zero
        self.p = [[0] * (cols + 1) for _ in range(rows + 1)]
        for r in range(rows):
            for c in range(cols):
                # cell + up block + left block - overlap counted twice
                self.p[r + 1][c + 1] = (
                    matrix[r][c] + self.p[r][c + 1] + self.p[r + 1][c] - self.p[r][c]
                )

    def sum_region(self, r1, c1, r2, c2):
        """Sum of matrix[r1..r2][c1..c2]."""
        p = self.p
        # whole - above - left + corner
        return p[r2 + 1][c2 + 1] - p[r1][c2 + 1] - p[r2 + 1][c1] + p[r1][c1]


m = NumMatrix(
    [
        [3, 0, 1, 4, 2],
        [5, 6, 3, 2, 1],
        [1, 2, 0, 1, 5],
        [4, 1, 0, 1, 7],
        [1, 0, 3, 0, 5],
    ]
)
assert m.sum_region(2, 1, 4, 3) == 8
assert m.sum_region(1, 1, 2, 2) == 11
assert m.sum_region(1, 2, 2, 4) == 12
print(m.sum_region(2, 1, 4, 3))

8


### 4.2.2 [1109. Corporate Flight Bookings](https://leetcode.com/problems/corporate-flight-bookings/) — ★ · Medium

**Problem.** Bookings `[first, last, seats]` add seats to a range of flights; return totals per flight.  
**Example.** `bookings = [[1,2,10],[2,3,20],[2,5,25]], n = 5` → `[10, 55, 45, 25, 25]`  
**Spot it.** Many range-add updates, one final read → difference array.

**Ask first**
- 1-indexed and inclusive? → Yes, both — get the `last + 1` index right.

> 💡 **Idea.** Difference array: for a booking `[first, last, seats]` (1-indexed flights) do `diff[first − 1] += seats` and `diff[last] −= seats`. One prefix sum at the end rebuilds every flight's total.

**Brute force:** add seats to every flight in each range, O(n·bookings).

**Steps**
1. `diff[first − 1] += seats`, `diff[last] −= seats` for each booking.
2. Prefix-sum the difference array.
3. The running sums are the per-flight totals.

**Cost:** Time O(n + b) (O(1) per booking plus one pass) · Space O(n) (the difference array)

**Likely follow-ups**
- **Bookings and queries interleave (online)?** Fenwick tree on the difference array: range-add = two point updates, point query = prefix sum. O(log n) each.
- **Need range *sums* after range adds?** Two Fenwick trees (or a lazy segment tree).
- **Flight numbers up to 10⁹?** Sort the `+`/`−` events by position (coordinate compression) instead of a dense array.

In [54]:
def corp_flight_bookings(bookings, n):
    """Seat totals per flight after range bookings."""
    diff = [0] * (n + 1)
    for first, last, seats in bookings:
        # the range starts adding here...
        diff[first - 1] += seats
        # ...and stops adding after `last`
        diff[last] -= seats
    # a prefix sum over the changes rebuilds the totals
    out, running = [], 0
    for i in range(n):
        running += diff[i]
        out.append(running)
    return out


assert corp_flight_bookings([[1, 2, 10], [2, 3, 20], [2, 5, 25]], 5) == [
    10,
    55,
    45,
    25,
    25,
]
assert corp_flight_bookings([[1, 2, 10], [2, 2, 15]], 2) == [10, 25]
print(corp_flight_bookings([[1, 2, 10], [2, 3, 20], [2, 5, 25]], 5))

[10, 55, 45, 25, 25]


### 4.2.3 [1094. Car Pooling](https://leetcode.com/problems/car-pooling/) — ★★ · Medium

**Problem.** A car with `capacity` seats drives east; each trip `[passengers, start, end]` gets in at `start` and out at `end`. Can the car complete every trip?  
**Example.** `trips = [[2, 1, 5], [3, 3, 7]], capacity = 4` → `False` (5 people ride between 3 and 5)  
**Spot it.** Many range additions, then "is any point over the limit?" → difference array, then scan for the peak.

**Ask first**
- Do passengers leave *before* new ones board at the same stop? → Yes — so `−p` at `to` (not `to + 1`).
- Locations ≤ 1000? → Yes → array. Huge coordinates → sort the events instead.

> 💡 **Idea.** Each trip adds `p` passengers at `from` and removes them at `to`. Mark `+p`/`−p` in a difference array, sweep left to right, and fail if the running load ever beats capacity.

**Brute force:** for every location, add up all trips covering it, O(range·trips).

**Steps**
1. `change[start] += p` and `change[end] −= p` for each trip.
2. Walk the points left to right with a running load.
3. Return `False` as soon as the load exceeds `capacity`.

**Cost:** Time O(n + L) (one update per trip, one pass over L points) · Space O(L) (the change array)

**Likely follow-ups**
- **Locations up to 10⁹?** Make events `(pos, +p)` and `(pos, −p)`, sort (drop-offs before pick-ups at equal pos), sweep. O(n log n).
- **What's the minimum capacity needed?** Same sweep — return the max running load.
- **Several cars?** Sort trips by start; min-heap of cars by free time (like Meeting Rooms II), or check if peak load ≤ total capacity when passengers can be split.

In [55]:
def car_pooling(trips, capacity):
    """True if the car never holds more than `capacity` passengers."""
    last = max(end for _, _, end in trips)
    change = [0] * (last + 1)
    for passengers, start, end in trips:
        # passengers get in at start...
        change[start] += passengers
        # ...and are out of the car at end
        change[end] -= passengers
    # running sum = passengers in the car at each point
    load = 0
    for delta in change:
        load += delta
        if load > capacity:
            return False
    return True


assert car_pooling([[2, 1, 5], [3, 3, 7]], 4) is False
assert car_pooling([[2, 1, 5], [3, 3, 7]], 5) is True
assert car_pooling([[2, 1, 5], [3, 5, 7]], 3) is True
print(car_pooling([[2, 1, 5], [3, 3, 7]], 4))

False


### 4.2.4 [2536. Increment Submatrices by One](https://leetcode.com/problems/increment-submatrices-by-one/) — ★★ · Medium

**Problem.** Start with an `n × n` grid of zeros; each query `[r1, c1, r2, c2]` adds 1 to every cell of that rectangle. Return the final grid.  
**Example.** `n = 3, queries = [[1, 1, 2, 2], [0, 0, 1, 1]]` → `[[1, 1, 0], [1, 2, 1], [0, 1, 1]]`  
**Spot it.** Many rectangle updates, one final read → 2D difference array.

**Ask first**
- Inclusive corners? → Yes → the `−1`s go at `r2 + 1` and `c2 + 1` (guard the bounds or pad the grid).
- Lots of queries on a big grid? → Yes, so filling cells one by one (up to 2.5·10⁹ steps) is too slow.

> 💡 **Idea.** 2D difference array: each rectangle touches only 4 corners (`+1` top-left, `−1` right of it, `−1` below it, `+1` diagonal). Two prefix sums (rows then columns) rebuild the grid.

**Brute force:** add 1 to every cell of every rectangle, O(q·n²).

**Steps**
1. Make an `(n + 1) × (n + 1)` grid `diff` of zeros.
2. For each query, add the four corner marks.
3. Prefix-sum every row, then every column; the top-left `n × n` block is the answer.

**Cost:** Time O(q + n²) (four marks per query, two passes over the grid) · Space O(n²) (the difference grid)

**Likely follow-ups**
- **Which point is covered by the most rectangles?** Build the same grid, return its max. (Reported as the follow-up to a meeting-rooms question.)
- **Coordinates up to 10⁹?** Compress the distinct x's and y's, run the 2D difference on the compressed grid.
- **Queries interleave with point reads?** 2D Fenwick on the difference array: O(log² n) per op.
- **Simpler middle ground?** One 1D difference array per row: O(q·n + n²).

In [56]:
def range_add_queries(n, queries):
    """Final n × n grid after adding 1 inside each query rectangle."""
    diff = [[0] * (n + 1) for _ in range(n + 1)]
    for r1, c1, r2, c2 in queries:
        # four corner marks fence in the rectangle
        diff[r1][c1] += 1
        diff[r1][c2 + 1] -= 1
        diff[r2 + 1][c1] -= 1
        diff[r2 + 1][c2 + 1] += 1
    # prefix sums along each row
    for r in range(n + 1):
        for c in range(1, n + 1):
            diff[r][c] += diff[r][c - 1]
    # then down each column
    for r in range(1, n + 1):
        for c in range(n + 1):
            diff[r][c] += diff[r - 1][c]
    return [row[:n] for row in diff[:n]]


def range_add_slow(n, queries):
    """Reference: fill every cell of every rectangle."""
    grid = [[0] * n for _ in range(n)]
    for r1, c1, r2, c2 in queries:
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                grid[r][c] += 1
    return grid


assert range_add_queries(3, [[1, 1, 2, 2], [0, 0, 1, 1]]) == [
    [1, 1, 0],
    [1, 2, 1],
    [0, 1, 1],
]
assert range_add_queries(2, [[0, 0, 1, 1]]) == [[1, 1], [1, 1]]
rects = []
for _ in range(30):
    r1, r2 = sorted(random.sample(range(6), 2))
    c1, c2 = sorted(random.sample(range(6), 2))
    rects.append([r1, c1, r2, c2])
assert range_add_queries(6, rects) == range_add_slow(6, rects)
print(range_add_queries(3, [[1, 1, 2, 2], [0, 0, 1, 1]]))

[[1, 1, 0], [1, 2, 1], [0, 1, 1]]


### 4.2.5 [528. Random Pick with Weight](https://leetcode.com/problems/random-pick-with-weight/) — ★★ · Medium

**Problem.** Pick index `i` with probability `w[i] / sum(w)`.  
**Example.** `w = [1, 3]` → `pick()` returns index `1` about 75% of the time, index `0` about 25%  
**Spot it.** Sampling by weight → prefix sums + binary search on a uniform random number.

**Ask first**
- Many picks? → Yes, so precompute prefix sums once and binary-search each pick (O(log n)).

> 💡 **Idea.** Lay the weights end to end on a number line (prefix sums). Pick a random point and binary-search which segment it lands in.

**Brute force:** build a list with each index repeated `w[i]` times, O(sum of weights) memory.

**Steps**
1. Build the running prefix sums.
2. Draw an integer `target` in `[0, total)`.
3. Binary search for the first prefix greater than `target`.
4. Return that index.

**Cost:** Time O(n) build, O(log n) per pick · Space O(n)

**Likely follow-ups**
- **Weights change between picks?** Fenwick tree over the weights; descend the tree to find the segment. O(log n) for both update and pick.
- **Need O(1) per pick?** Alias method: O(n) setup, then pick a column and flip a biased coin. Bad if weights change.
- **Pick without replacement?** Remove the picked weight with a Fenwick update, or use the key trick: sort by `random() ** (1 / w)` and take the top k.

In [57]:
class WeightedPicker:
    """Sample indices proportional to weights."""

    def __init__(self, w):
        # prefix[i] = right edge of index i's segment
        self.prefix = []
        running = 0
        for x in w:
            running += x
            self.prefix.append(running)

    def pick_index(self):
        """Random index drawn by weight."""
        target = random.randrange(self.prefix[-1])
        # first segment whose right edge is beyond target
        lo, hi = 0, len(self.prefix) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if self.prefix[mid] > target:
                hi = mid
            else:
                lo = mid + 1
        return lo


random.seed(0)
picker = WeightedPicker([1, 3])
freq = Counter(picker.pick_index() for _ in range(10_000))
assert 0.70 < freq[1] / 10_000 < 0.80
print(freq)

Counter({1: 7500, 0: 2500})


---
# 5) Matrix

**The gist.** Matrix questions are index bookkeeping. Name the boundaries (`top, bottom, left, right`), move them one at a time, and when you must work in place, borrow spare space inside the matrix (first row/column, spare bits).

> **Think of it like…** peeling an onion: take the outer ring, then the next one in. Each ring is four straight walks.

**Handy identities:** clockwise rotation = transpose + reverse each row · counter-clockwise = reverse each row + transpose · diagonal key `r − c`, anti-diagonal key `r + c`.

**Classic bugs:** reading a single leftover row/column twice in a spiral · `[[0] * n] * m` (m copies of the same row) · updating cells while neighbours still need the old value.

## 5.1 Traversal and in-place transforms

Two tricks repeat: shrink four boundaries for layer-by-layer walks, and hide extra state *inside* the matrix (first row/column as flags, a second bit for the next state) to stay O(1) space.

### 5.1.1 [54. Spiral Matrix](https://leetcode.com/problems/spiral-matrix/) — ★★★ · Medium

**Problem.** Return all elements in spiral order.  
**Example.** `[[1,2,3],[4,5,6],[7,8,9]]` → `[1, 2, 3, 6, 9, 8, 7, 4, 5]`  
**Spot it.** Layer-by-layer walk with four shrinking boundaries.

**Ask first**
- Non-square, single row or single column? → Yes → check `top <= bottom` / `left <= right` before the left and up walks, or you read cells twice.

> 💡 **Idea.** Keep four walls `top, bottom, left, right`. Walk right, down, left, up, pulling each wall in after you walk it.

**Brute force:** simulate with a direction array and a `visited` grid, O(m·n) extra space.

**Steps**
1. Set `top, bottom, left, right`.
2. Read the top row left→right, then the right column top→bottom.
3. If a bottom row remains, read it right→left; if a left column remains, read it bottom→top.
4. Shrink all four boundaries; repeat while `top ≤ bottom` and `left ≤ right`.

**Cost:** Time O(mn) (every cell is read once) · Space O(1) extra besides the output

**Likely follow-ups**
- **Fill an n×n matrix with 1..n² in spiral order (LC 59)?** Same walls, write a counter instead of reading.
- **Counter-clockwise, or start from the centre?** Change the direction order (down, right, up, left); from the centre, walk 1, 1, 2, 2, 3, 3… steps turning each time (LC 885).
- **O(1) extra space?** The wall version already is (besides the output).

In [58]:
def spiral_order(matrix):
    """Elements in clockwise spiral order."""
    out = []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1
    
    while top <= bottom and left <= right:
        # top row, left to right
        out += [matrix[top][c] for c in range(left, right + 1)]
        # right column, downward
        out += [matrix[r][right] for r in range(top + 1, bottom + 1)]
        # bottom row, right to left (only if it is a different row)
        if top < bottom:
            out += [matrix[bottom][c] for c in range(right - 1, left - 1, -1)]
        # left column, upward (only if it is a different column)
        if left < right:
            out += [matrix[r][left] for r in range(bottom - 1, top, -1)]
        # peel the layer
        top, bottom, left, right = top + 1, bottom - 1, left + 1, right - 1
    return out


assert spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5]
assert spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]) == [
    1,
    2,
    3,
    4,
    8,
    12,
    11,
    10,
    9,
    5,
    6,
    7,
]
assert spiral_order([[1], [2], [3]]) == [1, 2, 3]
print(spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))

[1, 2, 3, 6, 9, 8, 7, 4, 5]


### 5.1.2 [48. Rotate Image](https://leetcode.com/problems/rotate-image/) — ★★★ · Medium

**Problem.** Rotate an `n × n` matrix 90° clockwise in place.  
**Example.** `[[1,2,3],[4,5,6],[7,8,9]]` → `[[7,4,1],[8,5,2],[9,6,3]]`  
**Spot it.** Rotation = transpose + reverse each row.

**Ask first**
- In place? → Yes, no second matrix.

> 💡 **Idea.** Clockwise 90° = transpose (flip over the diagonal) + reverse each row. Both are in-place swaps.

**Brute force:** copy into a new matrix with `new[c][n−1−r] = old[r][c]`, O(n²) extra.

**Steps**
1. Transpose: swap `m[i][j]` and `m[j][i]` for `j > i`.
2. Reverse every row.
3. (Counter-clockwise: reverse rows first, then transpose.)

**Cost:** Time O(n²) (each cell moves a constant number of times) · Space O(1)

**Likely follow-ups**
- **Counter-clockwise?** Transpose + reverse each column (or reverse each row, then transpose).
- **180°?** Reverse the row order, then reverse each row.
- **Non-square m×n?** Can't be done in place (shape changes to n×m) — build a new matrix.
- **Rotate layer by layer instead?** For each ring, rotate 4 cells at a time: `top → right → bottom → left → top`.

In [59]:
def rotate_image(matrix):
    """Rotate square matrix clockwise in place."""
    n = len(matrix)
    # transpose: mirror across the main diagonal
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    # mirror each row left-right
    for row in matrix:
        row.reverse()


m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
rotate_image(m)
assert m == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
print(m)

[[7, 4, 1], [8, 5, 2], [9, 6, 3]]


### 5.1.3 [73. Set Matrix Zeroes](https://leetcode.com/problems/set-matrix-zeroes/) — ★★★ · Medium

**Problem.** If a cell is 0, set its whole row and column to 0, in place with O(1) space.  
**Example.** `[[1,1,1],[1,0,1],[1,1,1]]` → `[[1,0,1],[0,0,0],[1,0,1]]`  
**Spot it.** Need markers without extra memory → use the first row and column as marker storage.

**Ask first**
- O(1) extra space required? → If not, two sets of zero rows/cols (O(m + n)) is much simpler.

> 💡 **Idea.** Use the first row and first column as the "zero this line" flags. Remember separately whether row 0 / col 0 had their own zeros, then clear the inside, then the first row/col.

**Brute force:** copy the matrix and zero lines from the copy, O(m·n) extra.

**Steps**
1. Record whether row 0 and column 0 originally contain a zero.
2. For every inner zero, set `m[r][0]` and `m[0][c]` to 0.
3. Zero every inner cell whose row or column flag is 0.
4. Finally zero row 0 and/or column 0 if their own flags were set.

**Cost:** Time O(mn) (a constant number of passes) · Space O(1) (two booleans)

**Likely follow-ups**
- **Why the separate flags for row 0 / col 0?** Their cells are used as markers, so you can't tell an original zero from a marker without remembering it first.
- **Matrix is huge and sparse?** Store only the zero positions, collect their rows/cols, and answer "is (r, c) zero?" from those sets.

In [60]:
def set_zeroes(matrix):
    """Zero rows/columns of every zero cell, O(1) space."""
    rows, cols = len(matrix), len(matrix[0])
    # remember row 0 / column 0 before they are used as flags
    first_row = any(matrix[0][c] == 0 for c in range(cols))
    first_col = any(matrix[r][0] == 0 for r in range(rows))
    # flag the row and column of every inner zero
    for r in range(1, rows):
        for c in range(1, cols):
            if matrix[r][c] == 0:
                matrix[r][0] = matrix[0][c] = 0
    # apply the flags to the inner cells
    for r in range(1, rows):
        for c in range(1, cols):
            if matrix[r][0] == 0 or matrix[0][c] == 0:
                matrix[r][c] = 0
    # finally handle the flag row and column themselves
    if first_row:
        matrix[0] = [0] * cols
    if first_col:
        for r in range(rows):
            matrix[r][0] = 0


m = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]
set_zeroes(m)
assert m == [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]
m = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]
set_zeroes(m)
assert m == [[1, 0, 1], [0, 0, 0], [1, 0, 1]]
print(m)

[[1, 0, 1], [0, 0, 0], [1, 0, 1]]


### 5.1.4 [289. Game of Life](https://leetcode.com/problems/game-of-life/) — ★★ · Medium

**Problem.** Compute the next state of Conway's Game of Life in place.  
**Example.** `[[0,1,0],[0,0,1],[1,1,1],[0,0,0]]` → `[[0,0,0],[1,0,1],[0,1,1],[0,1,0]]`  
**Spot it.** Need old and new state at once → encode both in each cell's bits.

**Ask first**
- In place? → Yes → two states per cell (bit trick or codes like 2 = dying, 3 = born). Otherwise just copy the board.

> 💡 **Idea.** Every cell must update from the *old* board. Store the next state in a second bit (`cell |= next << 1`), then shift everything right by 1 at the end.

**Brute force:** copy the board, compute from the copy, O(m·n) extra.

**Steps**
1. For each cell, count live neighbours using `& 1`.
2. If it lives next round (3 neighbours, or 2 and currently alive), set bit 1.
3. After all cells, shift every cell right by one to make "next" the new "now".

**Cost:** Time O(mn) (8 neighbours per cell) · Space O(1)

**Likely follow-ups**
- **Board is infinite / mostly empty?** Keep a set of live cells. Count neighbours only around live cells with a `Counter`; a cell lives if count == 3, or 2 and already alive. Cost ∝ live cells.
- **Board too big for memory but rows stream in?** Process three rows at a time (previous, current, next).
- **Run many generations fast?** Only re-check cells next to something that changed; or use hashing of repeating blocks (Hashlife).

In [61]:
def game_of_life(board):
    """Advance the board one generation in place."""
    rows, cols = len(board), len(board[0])
    for r in range(rows):
        for c in range(cols):
            # bit 0 still holds every neighbour's current state
            live = sum(
                board[i][j] & 1
                for i in range(max(0, r - 1), min(rows, r + 2))
                for j in range(max(0, c - 1), min(cols, c + 2))
                if (i, j) != (r, c)
            )
            # survives or is born: store the next state in bit 1
            if live == 3 or (live == 2 and board[r][c] & 1):
                board[r][c] |= 2
    # next state becomes the current state
    for r in range(rows):
        for c in range(cols):
            board[r][c] >>= 1


b = [[0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 0, 0]]
game_of_life(b)
assert b == [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]]
print(b)

[[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]]


---
# 6) Strings

**The gist.** Most string questions are careful simulation: follow the rules exactly and nail the edge cases. A few hide a trick (a custom sort order).

> **Think of it like…** reading a hand-filled form: skip blanks, read the sign, read digits until something isn't a digit, stop.

**Spot it:** "parse", "convert", "valid…", "compress", "compare versions", a precise list of rules (atoi), "largest number formed by" (custom comparator).

**Edge cases to list first:** empty / all spaces · sign and leading zeros (`"-042"`) · overflow clamping (Python never overflows — clamp yourself) · use a list + `"".join`, not `+=` in a loop.

## 6.1 Parsing and simulation

Write the edge cases down before coding — here that list *is* most of the solution.

### 6.1.1 [6. Zigzag Conversion](https://leetcode.com/problems/zigzag-conversion/) — ★★ · Medium

**Problem.** Write `s` in a zigzag over `num_rows` rows, then read row by row.  
**Example.** `s = "PAYPALISHIRING", num_rows = 3` → `"PAHNAPLSIIGYIR"`  
**Spot it.** Simulate with a row index that bounces between 0 and `num_rows − 1`.

**Ask first**
- `numRows` = 1 or ≥ len(s)? → Return `s` unchanged (otherwise the bounce breaks).

> 💡 **Idea.** Walk the string bouncing a row index 0 → numRows−1 → 0…, appending each char to its row. Join the rows.

**Brute force:** build the full 2D zigzag grid and read it, O(n·rows) space.

**Steps**
1. One accumulator per row; `row = 0`, `step = +1`.
2. Append each character to `rows[row]`.
3. Flip `step` at the top and bottom rows; move `row += step`.
4. Join the rows.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **O(1) extra space?** Read row by row with the cycle `2·(rows − 1)`: row `r` takes chars at `k·cycle + r` and (middle rows) `(k+1)·cycle − r`.
- **Reverse it — decode a zigzag string?** Compute how many chars each row gets from the cycle, slice them out, then bounce through rows to rebuild.

In [ ]:
def convert(s, num_rows):
    """Zigzag row-by-row reading of s."""
    if num_rows == 1:
        return s
    
    rows = [""] * num_rows
    row, step = 0, 1
    for ch in s:
        rows[row] += ch
        # bounce at the top and bottom rows
        if row == 0:
            step = 1
        elif row == num_rows - 1:
            step = -1
        row += step
    return "".join(rows)


assert convert("PAYPALISHIRING", 3) == "PAHNAPLSIIGYIR"
assert convert("PAYPALISHIRING", 4) == "PINALSIGYAHRPI"
assert convert("A", 1) == "A"
print(convert("PAYPALISHIRING", 3))

PAHNAPLSIIGYIR


### 6.1.2 [8. String to Integer (atoi)](https://leetcode.com/problems/string-to-integer-atoi/) — ★★ · Medium

**Problem.** Parse an int: skip leading spaces, optional sign, digits until a non-digit, clamp to 32-bit.  
**Example.** `"   -42"` → `-42`; `"4193 with words"` → `4193`  
**Spot it.** Step-by-step parser; clamp at the end (Python has no overflow).

**Ask first**
- What happens on overflow? → Clamp to `[−2³¹, 2³¹ − 1]`.
- No digits at all, or `+-12`? → Return 0.

> 💡 **Idea.** Follow the rules in order: skip spaces, read one optional sign, read digits until something isn't a digit, clamp to 32-bit.

**Brute force:** regex `^\s*[+-]?\d+` — fine to mention, but they want the manual parse.

**Steps**
1. Skip leading spaces.
2. Read an optional `+` or `−`.
3. Accumulate digits until a non-digit.
4. Apply the sign and clamp to `[−2³¹, 2³¹ − 1]`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Detect overflow without a bigger int type?** Before `res = res*10 + d`, check `res > MAX // 10 or (res == MAX // 10 and d > 7)` (8 for the negative side).
- **Parse floats / scientific notation (LC 65)?** A small state machine: sign → digits → optional `.` digits → optional `e` sign digits.
- **Return an error instead of clamping?** Raise/return an error code — ask which behaviour they want.

In [ ]:
def my_atoi(s):
    """C-style atoi with 32-bit clamping."""
    i, n = 0, len(s)

    # leading spaces
    while i < n and s[i] == " ":
        i += 1
    
    # optional sign
    sign = 1
    if i < n and s[i] in "+-":
        sign = -1 if s[i] == "-" else 1
        i += 1
    
    # digits until the first non-digit
    value = 0
    while i < n and s[i].isdigit():
        value = value * 10 + int(s[i])
        i += 1
    # clamp to the 32-bit range
    return max(-(2**31), min(2**31 - 1, sign * value))


assert my_atoi("42") == 42
assert my_atoi("   -042") == -42
assert my_atoi("1337c0d3") == 1337
assert my_atoi("words 987") == 0
assert my_atoi("-91283472332") == -(2**31)
print(my_atoi("   -042"))

-42


### 6.1.3 [443. String Compression](https://leetcode.com/problems/string-compression/) — ★ · Medium

**Problem.** Compress runs in place (`aabccc` → `a2bc3`); return the new length.  
**Example.** `["a", "a", "b", "b", "c", "c", "c"]` → returns `6`, and the list starts with `["a", "2", "b", "2", "c", "3"]`  
**Spot it.** Run-length encoding with read and write pointers.

**Ask first**
- Counts ≥ 10? → Write each digit (`a12`).
- Is a count of 1 written? → No, just the letter.

> 💡 **Idea.** Read runs with one pointer, write `char` + count digits with another. The output is never longer than the input, so writing in place is safe.

**Brute force:** build a new string, O(n) space.

**Steps**
1. At `read`, find the end of the current run.
2. Write the character.
3. If the run is longer than 1, write each digit of its length.
4. Continue from the run's end; return `write`.

**Cost:** Time O(n) · Space O(1)

**Likely follow-ups**
- **Decompress it again?** Read a char, then read all following digits as the count, expand. Nested like `3[ab]` needs a stack (LC 394).
- **What if compression makes it longer (e.g. "abc")?** Here count 1 isn't written so it never grows. In the CTCI variant, return the original if the result isn't shorter.
- **Stream that's too big for memory?** Process it chunk by chunk, carrying the current `(char, count)` across chunk borders.

In [ ]:
def compress(chars):
    """In-place run-length encode; return new length."""
    # start: where the current run begins; write: where the next output character goes
    start = 0
    write = 0

    while start < len(chars):
        ch = chars[start]

        # move `end` to just past the last copy of ch in this run
        end = start
        while end < len(chars) and chars[end] == ch:
            end += 1
        run_length = end - start

        # write the character once
        chars[write] = ch
        write += 1

        # write the count only if the run is longer than 1 (digit by digit: 12 -> "1", "2")
        if run_length > 1:
            for digit in str(run_length):
                chars[write] = digit
                write += 1

        # the next run starts where this one ended
        start = end

    return write


c = list("aabbccc")
k = compress(c)
assert c[:k] == list("a2b2c3")
c = list("abbbbbbbbbbbb")
assert c[: compress(c)] == list("ab12")
print(k)

6


### 6.1.4 [179. Largest Number](https://leetcode.com/problems/largest-number/) — ★★ · Medium

**Problem.** Arrange non-negative integers to form the largest number (as a string).  
**Example.** `[3, 30, 34, 5, 9]` → `"9534330"`  
**Spot it.** Custom ordering: `a` before `b` iff `a + b > b + a` as strings.

**Ask first**
- Return a string? → Yes, the number can be huge.
- All zeros? → Return `"0"`, not `"000"`.

> 💡 **Idea.** Sort as strings with a custom rule: `a` goes before `b` if `a + b > b + a`. Then join. Strip the all-zeros case to `"0"`.

**Brute force:** try every order, O(n!).

**Steps**
1. Convert numbers to strings.
2. Sort so that `a` comes before `b` when `a + b > b + a` (compare neighbours and swap if wrong).
3. Join; if the first piece is `"0"`, the answer is `"0"`.

**Cost:** Time O(n² · k) with the simple swap sort (each comparison joins two k-digit strings; O(n log n · k) with Python's `sort`) · Space O(n · k)

**Likely follow-ups**
- **Why is that comparator a valid sort order?** It's transitive: `ab > ba` and `bc > cb` ⇒ `ac > ca` (think of each as a fraction `a / (10^len(a) − 1)`).
- **Smallest number instead?** Flip the rule (`a + b < b + a`) and strip leading zeros (keep one `0` if all zeros).
- **Python sort with a comparator?** `sorted(strs, key=cmp_to_key(lambda a, b: -1 if a + b > b + a else 1))`.

In [ ]:
def largest_number(nums):
    """Largest concatenation of the numbers."""
    strs = [str(x) for x in nums]
    n = len(strs)

    # simple sort: repeatedly swap neighbours that are in the wrong order
    for i in range(n):
        for j in range(n - 1 - i):
            a = strs[j]
            b = strs[j + 1]
            # b should go first when "b then a" beats "a then b"
            if b + a > a + b:
                strs[j] = b
                strs[j + 1] = a

    # all zeros collapse to a single "0"
    if strs[0] == "0":
        return "0"
    return "".join(strs)


assert largest_number([10, 2]) == "210"
assert largest_number([3, 30, 34, 5, 9]) == "9534330"
assert largest_number([0, 0]) == "0"
print(largest_number([3, 30, 34, 5, 9]))

9534330


### 6.1.5 [165. Compare Version Numbers](https://leetcode.com/problems/compare-version-numbers/) — ★ · Medium

**Problem.** Compare version strings like `1.01` vs `1.001.0`; return −1, 0 or 1.  
**Example.** `"1.01"` vs `"1.001"` → `0` (equal)  
**Spot it.** Split on dots, compare integer revisions, pad the shorter with zeros.

**Ask first**
- Leading zeros? → Ignored (`"01" == "1"`) — compare as ints, not strings.
- Missing revision? → Counts as 0 (`"1.0" == "1"`).

> 💡 **Idea.** Split on `.`, compare revision by revision as integers, treat missing ones as 0.

**Brute force:** string compare — wrong (`"10" < "9"` as strings).

**Steps**
1. Split both versions on `.` and convert each part to `int`.
2. Walk them in parallel, padding the shorter with 0.
3. Return at the first difference; equal throughout → 0.

**Cost:** Time O(m + n) · Space O(m + n) for the split parts

**Likely follow-ups**
- **O(1) extra space (no split)?** Two pointers; parse each revision number on the fly up to the next `.`.
- **Support semver pre-release tags (`1.0.0-alpha < 1.0.0`)?** Compare the numeric core first; a version *with* a pre-release tag is lower; compare tags piece by piece (numbers as ints, text lexically).
- **Sort a list of versions?** `sorted(vs, key=lambda v: [int(x) for x in v.split('.')])` — but pad with zeros so `1.0` and `1` tie.

In [ ]:
def compare_version(version1, version2):
    """Sign of version1 - version2."""
    # split into revision numbers; int() drops leading zeros ("01" -> 1)
    parts1 = [int(p) for p in version1.split(".")]
    parts2 = [int(p) for p in version2.split(".")]

    # compare as many revisions as the longer version has
    for i in range(max(len(parts1), len(parts2))):
        # a missing revision counts as 0
        x = parts1[i] if i < len(parts1) else 0
        y = parts2[i] if i < len(parts2) else 0

        if x > y:
            return 1
        if x < y:
            return -1
    return 0


assert compare_version("1.2", "1.10") == -1
assert compare_version("1.01", "1.001") == 0
assert compare_version("1.0", "1.0.0.0") == 0
print(compare_version("1.2", "1.10"))

-1
